# Module 4 — Building the Graph

*Live session, 1.5h. Run alongside the discussion.*

Four public datasets went in. One graph came out. Along the way somebody decided
which rows describe the same company — and that decision is now a fact the rest
of the course reasons over.

Today: how to find out which of those decisions are wrong, **without being told
the answer**.

In [3]:
import json
import os

from IPython.display import SVG, JSON, Markdown, display
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.environ.get("NEO4J_USER", "neo4j"),
          os.environ.get("NEO4J_PASSWORD", "password")))

# The database is named, not left to the server default. The default database on
# a Neo4j instance that has been used for anything else holds another graph, and
# asking the wrong one does not raise: the queries below return, with numbers
# from whatever is in there. Checked 21/10/2026: the default here answered with
# 1,112,579 nodes and no `Document` label at all.
DATABASE = os.environ.get("NEO4J_DATABASE", "kgtp")

def q(cypher, **params):
    with driver.session(database=DATABASE) as s:
        return [dict(r) for r in s.run(cypher, **params)]

def xml(text):
    """What SVG cannot swallow. One vendor is called `A & M AMUSEMENT INC`, and
    an unescaped ampersand does not draw badly -- it stops the drawing."""
    return (str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))

def fit(text, width, size):
    """Shorten a label to what its box can hold, at roughly 0.55 em per
    character. Without it a long department name draws straight out of its
    box and over whatever is next to it."""
    room = int((width - 16) / (size * 0.55))
    return text if len(text) <= room else text[:room - 1] + "\u2026"

def show_json(raw, mode="text", indent=2):
    """Display a result as JSON instead of a one-line Python repr.

    `default=str` is not cosmetic: Neo4j returns its own temporal types, and
    `json.dumps` raises TypeError on any result carrying a date.
    """
    if isinstance(raw, str):
        raw = json.loads(raw)
    if mode in ("tree", "both"):
        display(JSON(raw))
    if mode in ("text", "both"):
        display(Markdown("```json\n"
                         + json.dumps(raw, indent=indent, ensure_ascii=False, default=str)
                         + "\n```"))

In [4]:
show_json(q('''
RETURN COUNT { MATCH (n) RETURN n }        AS nodes,
       COUNT { MATCH ()-[r]->() RETURN r } AS relationships
'''))

```json
[
  {
    "nodes": 1112579,
    "relationships": 1529922
  }
]
```

## Four files, one corpus, and where they touch

Four public datasets from the City of Chicago portal went in, and so did the
scanned contracts themselves. You were given neither, so start from what the
graph kept: every record node carries a `source`, and it is the only thing that
survives to say where a row came from.

The portal's name for a file and the value stamped in the graph are not the same
word — the payroll answers to `EMPLOYEES` — and nothing in the graph says so:

| file | `source` | what its rows became |
|---|---|---|
| city contracts (`rsxa-ify5`) | `CONTRACTS` | `ContractRecord`, plus the `Organization` and `Department` each row names |
| business licences (`r5kz-chrr`) | `LICENSES` | `LicenseRecord`, plus one `Organization` per licensee |
| business owners (`ezma-pppn`) | `OWNERS` | `PersonRecord` — the owners named on those licences |
| city payroll (`xzkq-xp2w`) | `EMPLOYEES` | `PersonRecord`, plus the `Department` that pays them |

**The corpus is not a fifth dataset.** It is the first file in the other medium
the portal publishes: every contract row carries a viewer URL, and behind it sit
the scanned PDFs — **14,499 documents and 279,017 pages**, each one hanging off
the contract record that names it.

What `source` cannot say is what happened next. A CSV row became a record
because somebody loaded a column; a page became a record because somebody
**read** it. So this table groups by what reading produced rather than by which
form it was read from — the stamp on each node still names the form, `DOC_EDS`
and `DOC_B2GNOW` among eleven:

| what reading produced | | read from |
|---|---|---|
| `PersonRecord` | 10,891 | signature blocks and contact lines: the signatory of an EDS, the buyer on a modification, a commissioner, a procurement specialist |
| `Organization` | 1,269 | the firm a form is filed for, and the primes and subcontractors named on it |
| `Modification` | 2,312 | a Modification Summary Report, with the reason it gives |
| `ComplianceAudit` | 10 | a B2Gnow audit page |
| `ReasonCode`, `Certification` | 6, 6 | the two vocabularies those pages use |

**Two things to read off it.** The labels are the same labels. The corpus does
not arrive beside the graph, it arrives inside it: 10,891 more `PersonRecord`
and 1,269 more `Organization`, which are the two populations the rest of today
is about.

And the line between a record that was loaded and a record that was read is a
query, not a convention. Of those 14,494 nodes, **14,482 carry `doc` and
`page`** — and both resolve: every one of them names a `Document` and a `Page`
that the contracts file also put in the graph. **Not one node from the four
files carries either.** The twelve that do not are `ReasonCode` and
`Certification`, two vocabularies rather than records — the same shape as
`ContractType` on the other side.

`Page` itself carries no `source` at all: 279,017 nodes, zero. The paper is not
a source; reading it is.


In [ ]:
show_json(q('''
// Which sources name which records, by node type. `source` is stamped at load
// time, and for every label but one it is the whole answer. `Department` is the
// exception: both files MERGE on the same `id`, so one node can be named by
// both and a single stamp cannot hold that -- there the sources are read from
// the edges instead.
MATCH (n) WHERE n.source IN ['CONTRACTS', 'LICENSES', 'OWNERS', 'EMPLOYEES']
           OR n.source STARTS WITH 'DOC_'
WITH n, labels(n)[0] AS node,
     CASE WHEN n:Department
          THEN [s IN [CASE WHEN exists { (:PersonRecord {source: 'EMPLOYEES'})
                                         -[:WORKS_FOR_DEPARTMENT]->(n) }
                           THEN 'EMPLOYEES' END,
                      CASE WHEN exists { (:ContractRecord)
                                         -[:RECORDED_DEPARTMENT]->(n) }
                           THEN 'CONTRACTS' END]
                WHERE s IS NOT NULL]
          ELSE [n.source] END AS sources
UNWIND sources AS source
WITH node, source, count(*) AS records
ORDER BY records DESC
WITH node,
     collect(CASE WHEN NOT source STARTS WITH 'DOC_'
                  THEN {source: source, records: records} END) AS from_files,
     collect(CASE WHEN source STARTS WITH 'DOC_'
                  THEN {source: source, records: records} END) AS from_corpus
RETURN node, from_files, from_corpus
ORDER BY size(from_files) + size(from_corpus) DESC,
         (from_files + from_corpus)[0].records DESC
'''))


**Read the rows where one node type comes from two files.** Three do, and they
are the whole reason this graph exists:

- **`Organization`** — 196,646 from licences, 14,094 from contracts. A contract
  names a supplier, a licence names a business at an address, and **nothing in
  either file says they are the same firm.** Both sides do carry an address as
  well as a name, so there is more than one signal to decide with.
- **`PersonRecord`** — 329,934 from business owners, 31,760 from the payroll. An
  owner arrives with the company whose licence names them, an employee with the
  department that pays them, and **neither side carries what the other has.**
  The only field both write down is the name.
- **`Department`** — 39 named by the payroll, 72 by contracts, and 76 nodes to
  hold them: the two sides overlap on **35 bodies that both files name**. That
  junction was made at load time, for free, by string equality — the two
  vocabularies spell those 35 the same way — and it is the one place `source`
  cannot answer the question, because a node named by two files still carries
  one stamp.

Everywhere else a file contributes its own kind of record and nothing else.

Somebody decided which rows on the two sides of each join describe the same
thing, and those decisions are now facts the rest of the course reasons over.
Integration is not the plumbing before the interesting part — it is the part
where new facts appear, and where the wrong ones do.

Today takes two of the three joins. **People first**, because they are the
hardest: one field to match on and no second signal to decide with.
**Organisations after**, because there the address supplies that second
signal — which buys a great deal and breaks in a different way.

### What the two files actually give you

Before any matching, look at what you have. Everything that follows is forced by
this table.

People appear in **two** datasets: business owners, and city employees.

In [ ]:
show_json(q('''
// Scope: the two files that carry people, OWNERS and EMPLOYEES. The records
// read from documents never went through the narrowing step -- decision 65.
MATCH (p:PersonRecord) WHERE NOT p.source STARTS WITH 'DOC_'
OPTIONAL MATCH (p)-[:WORKS_FOR_DEPARTMENT]->(d:Department)
RETURN p.source            AS source,
       count(*)            AS records,
       count(p.employerId) AS with_employerId,
       count(d)            AS with_department,
       count(p.firstName)  AS with_firstName
ORDER BY records DESC
'''))

Read it column by column, because the shape is the whole point.

**Owners bring a company.** Every one of the 329,934 owner records points at the
business whose licence names them.

**Employees bring a department.** Every one of the 31,760 points at the city
department that pays them.

**Neither brings what the other has.** Not one owner record has a department;
not one employee record has a company. The two files describe people from two
sides of the same city and share nothing but the name.

That is not a gap someone forgot to fill. A business licence has no reason to
record a city department, and a payroll has no reason to record a licence.

### The names are not written the same way either

Two real rows, one per file, as they are in the source. You do not have the
CSVs, and this is the only place in the session you see them.

**Payroll** — one column, `LAST, GIVEN MIDDLE` in a single string:

```
Name,Job Titles,Department,...
"RIORDAN, KEVIN C",PREPRESS TECHNICIAN,DEPARTMENT OF FLEET AND FACILITY MANAGEMENT,...
```

**Business owners** — first, middle initial and last in separate columns:

```
Account Number,Legal Name,Owner First Name,Owner Middle Initial,Owner Last Name,Suffix,Legal Entity Owner,Title
85613,MERCER (US) LLC,Ron,M,Anderson,,,OTHER
```

Both shapes had to become one string, `fullName` — **added**, not substituted.
The components survive in the casing their own file used, so
`"RIORDAN, KEVIN C"` is reconstructible from `lastName`, `firstName` and
`middleName`, and the owner columns are kept as they were written. The graph
does hold its origin.

What it does **not** hold is which row a record came from. The id is positional
— `4000000`, `3000001` — not something the source gave it. Reload a freshly
downloaded file whose rows have shifted, and that id points somewhere else.

Look at what was kept, and at how partial the normalisation was:

In [101]:
show_json(q('''
// Three per file, so both sides are visible: ORDER BY ... LIMIT would return
// only the first source. Restricted to records that have a middle name,
// because that is the component the two files disagree about most.
CALL () {
  MATCH (p:PersonRecord {source: 'EMPLOYEES'}) WHERE p.middleName IS NOT NULL RETURN p LIMIT 3
  UNION
  MATCH (p:PersonRecord {source: 'OWNERS'})    WHERE p.middleName IS NOT NULL RETURN p LIMIT 3
}
RETURN p.source     AS source,
       p.firstName  AS firstName,
       p.middleName AS middleName,
       p.lastName   AS lastName,
       p.fullName   AS fullName
ORDER BY source
'''))

```json
[
  {
    "source": "EMPLOYEES",
    "firstName": "KEVIN",
    "middleName": "C",
    "lastName": "RIORDAN",
    "fullName": "Kevin C Riordan"
  },
  {
    "source": "EMPLOYEES",
    "firstName": "ERICK",
    "middleName": "M",
    "lastName": "BRYANT",
    "fullName": "Erick M Bryant"
  },
  {
    "source": "EMPLOYEES",
    "firstName": "DANIEL",
    "middleName": "F",
    "lastName": "BUENAVENTURA",
    "fullName": "Daniel F Buenaventura"
  },
  {
    "source": "OWNERS",
    "firstName": "Ron",
    "middleName": "M",
    "lastName": "Anderson",
    "fullName": "Ron M Anderson"
  },
  {
    "source": "OWNERS",
    "firstName": "CELSO",
    "middleName": "RAFAEL",
    "lastName": "PERDOMO VARGAS",
    "fullName": "Celso Rafael Perdomo Vargas"
  },
  {
    "source": "OWNERS",
    "firstName": "CHARLES",
    "middleName": "H",
    "lastName": "KIM",
    "fullName": "Charles H Kim"
  }
]
```

`fullName` is consistent. The components underneath are not, and not even
within one file: the payroll is upper case throughout, while the owners file
writes `Ron M Anderson` on one row and `CELSO RAFAEL PERDOMO VARGAS` on the
next. Each component kept the casing of the row it came from.

### What this forces

Follow the consequences in order, because the rest of the session is each one in
turn:

1. people are in two files
2. owners have a company, employees have a department — **no overlap**
3. so the only field both sides carry is the name
4. and the two files write names differently, so they must be normalised first
5. therefore matching people is done **on names alone**

Step 5 is why there is exactly one similarity method in this graph:

In [105]:
show_json(q('''
MATCH (:PersonRecord)-[r:IS_SIMILAR_TO]->(:PersonRecord)
RETURN DISTINCT r.method AS method, count(*) AS edges
'''))


[{'method': 'SIMILAR_NAME', 'edges': 714058}]

One method, `SIMILAR_NAME`. The company is loaded and plays no part in
deciding who is who.

**That exclusion was a decision, and it is the one worth copying.** Before
building any similarity, list the signals the data actually carries and say, for
each, whether it will be used and why. Here the company was excluded because
only one of the two files has it, and a signal present on one side and absent on
the other cannot compare anything.

Hold on to that. It comes back twice: once as the thing that lets you choose a
threshold, and once as the reason you cannot check the pairs that matter most.

## Sixty-five billion pairs, and the two steps that cut them to four

You have seen what the four files give you, and that people are the join with
only one field to work with. Start from the fact underneath all of it.

**Two files describe people, and they share no identifier.** The only thing
both write down is a name. So matching has to happen on names — and comparing
every pair means:

In [106]:
# Same scope as the rest of the section -- see the note in the source query.
n = q("MATCH (p:PersonRecord) WHERE NOT p.source STARTS WITH 'DOC_' RETURN count(*) AS n")[0]["n"]
print(f"{n:,} records  ->  {n * (n - 1) // 2:,} pairs")

361,694 records  ->  65,411,093,971 pairs


Sixty-five billion. That is the constraint the whole pipeline answers to.

The answer is **two steps that look like one query and do different jobs**:

```cypher
-- 1. NARROW: which records are even worth comparing?
CALL db.index.fulltext.queryNodes(
       "person_record_fullName",
       apoc.text.join([x IN name_words | trim(x) + "~0.65"], " AND ")) YIELD node

-- 2. SCORE: of those, which are close enough to keep?
WITH p, node, apoc.text.sorensenDiceSimilarity(name, node.fullName) AS simil
WHERE simil > 0.695
```

`~0.65` is fuzzy matching **per word**, joined with `AND`: every word must find
an approximate match. It decides *who gets compared*.

`0.695` judges the whole string. It decides *who passes*.

Where does 0.695 come from? Nobody derived it. And the honest version of that
is not a shrug — it is the shape of the whole problem:

> There is no ground truth for these names, so the threshold cannot be
> optimised, only defended. A threshold can be shown wrong but never shown
> right.

Here, take the number as given and watch what it does to a single name.

### One name, end to end

Run the pipeline on a single person, with the threshold **removed**, so you see
everything the narrowing step proposed.

First the narrowing step on its own, because the number it hands back is not the
number you would expect.

In [ ]:
show_json(q('''
// Step 1 on its own: the full-text call, with the score the index returns.
MATCH (p:PersonRecord) WHERE p.fullName = $name
WITH p, apoc.text.split(apoc.text.replace(p.fullName, '[^a-zA-Z0-9\\s]', ''), "\\s+") AS name_words
CALL db.index.fulltext.queryNodes(
       "person_record_fullName",
       apoc.text.join([x IN name_words | trim(x) + "~0.65"], " AND ")) YIELD node, score
WITH p, node, score WHERE p <> node
WITH DISTINCT node, score
RETURN node.fullName AS candidate, round(score * 10000) / 10000 AS lucene_score
ORDER BY lucene_score DESC
''', name="Leo Appelbaum"))


**Four candidates, and a score that is not between 0 and 1.** `9.2238` and
`5.8099` are Lucene relevance: how well a document answers a query, built from
how rare each term is in the index. It has no ceiling, and it moves with the
query rather than with the pair — so there is no value of it on which a
threshold like 0.695 could be set.

And look at the two spellings of Lee: **the same score, 5.8099, for both.** The
index cannot separate them. The next cell can, at 0.737 and 0.700 — which is
the whole reason the pipeline scores a second time instead of trusting the step
that narrowed.


In [ ]:
CANDIDATES = """
MATCH (p:PersonRecord) WHERE p.fullName = $name
WITH p, p.fullName AS name,
     apoc.text.split(apoc.text.replace(p.fullName, '[^a-zA-Z0-9\\s]', ''), "\\s+") AS name_words
CALL db.index.fulltext.queryNodes(
       "person_record_fullName",
       apoc.text.join([x IN name_words | trim(x) + "~0.65"], " AND ")) YIELD node
WITH p, name, node WHERE p <> node
WITH DISTINCT node, apoc.text.sorensenDiceSimilarity(name, node.fullName) AS simil
// An employerId on its own says nothing. Follow it to the business, its address
// and its licence terms -- one row per period of validity, renewals
// included -- which is where the answer actually is.
MATCH (o:Organization {id: node.employerId})
OPTIONAL MATCH (o)-[:ORG_HAS_LICENSE]->(l:LicenseRecord)
WITH node, simil, o, count(l) AS licence_terms,
     // reordered to YYYY-MM-DD so that min and max sort as dates
     min(right(l.startDate, 4) + '-' + left(l.startDate, 2) + '-' + substring(l.startDate, 3, 2)) AS licensed_from,
     max(right(l.endDate, 4) + '-' + left(l.endDate, 2) + '-' + substring(l.endDate, 3, 2)) AS licensed_to
RETURN node.fullName AS candidate, round(simil * 1000) / 1000 AS simil,
       node.title AS role, node.employerId AS employerId,
       o.name AS business, o.addresses[0] AS address,
       licence_terms, licensed_from, licensed_to
ORDER BY simil DESC
"""

show_json(q(CANDIDATES, name="Leo Appelbaum"))


In [ ]:
SIMILARITIES = """
// Every similarity edge among those records, each pair once. `a.id < b.id` is
// what makes it once: the loader wrote some pairs in both directions.
MATCH (a:PersonRecord)-[r:IS_SIMILAR_TO]-(b:PersonRecord)
WHERE a.fullName IN $names AND b.fullName IN $names AND a.id < b.id
RETURN DISTINCT a.id AS one_id, a.fullName AS one, b.id AS other_id, b.fullName AS other,
       round(r.score * 1000) / 1000 AS score
ORDER BY one_id, score DESC
"""

NAMES = ["Leo Appelbaum", "Lee D Applbaum", "Lee D. Applbaum"]
EDGES = q(SIMILARITIES, names=NAMES)

# One node per record, not per name: `Leo Appelbaum` is written twice and the
# 1.000 between those two rows is an edge like any other. The id under each name
# is what tells them apart -- it is positional, and it means nothing outside
# this graph.
RECORDS = {}
for e in EDGES:
    RECORDS[e["one_id"]] = e["one"]
    RECORDS[e["other_id"]] = e["other"]

W, BW, BH, LW, LH = 780, 150, 42, 46, 18
SPOTS = dict(zip(sorted(RECORDS), [(280, 48), (104, 158), (456, 158), (280, 268)]))
# A group gets a pale fill for its records and the same hue, saturated, for the
# entity they resolve to: the eye carries the colour from one to the other.
GROUP_FILLS  = ["#CFD8DC", "#ACDBF8", "#ECEFF1"]
PERSON_FILLS = ["#607D8B", "#0091EA", "#9E9E9E"]

def draw(title=None, groups=None, persons=None):
    """The four records and the similarities between them, drawn from EDGES.

    `groups` colours a node by whatever grouping is passed in -- one fill per
    distinct value. Without it every node is the same colour, which is the
    picture of what the matcher said and nothing else.

    `persons` draws the entity each group resolves to, on the right, joined to
    its records by a dashed blue edge. A person marked `real=False` is a
    counterfactual: that grouping is not the one the graph kept, so no such
    node exists in it, and the dashed outline says so.
    """
    top = 40 if title else 0
    pos = {rid: (x, y + top) for rid, (x, y) in SPOTS.items()}
    fill = {}
    if groups:
        order = sorted({groups[r] for r in pos})
        fill = {r: GROUP_FILLS[order.index(groups[r]) % len(GROUP_FILLS)] for r in pos}

    def box(cx, cy, w=BW, h=BH):
        return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)

    def border(centre, target):
        """from a node's centre out to its own edge, aimed at the other node"""
        dx, dy = target[0] - centre[0], target[1] - centre[1]
        step = min((BW / 2) / abs(dx) if dx else 9e9, (BH / 2) / abs(dy) if dy else 9e9)
        return centre[0] + dx * step, centre[1] + dy * step

    def collide(a, b, pad=3):
        return not (a[2] + pad < b[0] or b[2] + pad < a[0]
                    or a[3] + pad < b[1] or b[3] + pad < a[1])

    def crosses(p, r, rect, pad=2):   # sampled; six segments do not need more
        return any(rect[0] - pad <= p[0] + (r[0] - p[0]) * i / 60 <= rect[2] + pad and
                   rect[1] - pad <= p[1] + (r[1] - p[1]) * i / 60 <= rect[3] + pad
                   for i in range(61))

    lines = [(border(pos[e["one_id"]], pos[e["other_id"]]),
              border(pos[e["other_id"]], pos[e["one_id"]]),
              f'{e["score"]:.3f}') for e in EDGES]

    # A label sits at the middle of its edge unless something is already there,
    # in which case it slides along the line until it is clear of every node,
    # every other label and every other edge.
    nodes, placed = [box(*p) for p in pos.values()], []
    for i, (start, end, label) in enumerate(lines):
        for t in (0.5, 0.36, 0.64, 0.28, 0.72, 0.22, 0.78):
            lx, ly = start[0] + (end[0] - start[0]) * t, start[1] + (end[1] - start[1]) * t
            here = box(lx, ly, LW, LH)
            if any(collide(here, n) for n in nodes) or any(collide(here, p) for p, _, _ in placed):
                continue
            if any(crosses(s, e, here) for j, (s, e, _) in enumerate(lines) if j != i):
                continue
            placed.append((here, (lx, ly), label))
            break

    # the entity column, one node per group, in the vertical order of the groups
    person_pos, person_box = {}, (150, 42)
    if persons:
        order = sorted({groups[r] for r in pos}, key=lambda g: min(pos[r][1] for r in pos if groups[r] == g))
        span = 110 if len(order) > 1 else 0
        for i, g in enumerate(order):
            person_pos[g] = (660, 158 + top + (i - (len(order) - 1) / 2) * span)

    height = 330 + top
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {height}" width="50%"'
           f' font-family="Arial, Helvetica, sans-serif">',
           f'<rect width="{W}" height="{height}" fill="#FFFFFF"/>']
    if title:
        svg.append(f'<text x="{W / 2}" y="26" font-size="13" font-weight="bold"'
                   f' fill="#0091EA" text-anchor="middle">{title}</text>')
    for start, end, _ in lines:
        svg.append(f'<line x1="{start[0]:.1f}" y1="{start[1]:.1f}" x2="{end[0]:.1f}"'
                   f' y2="{end[1]:.1f}" stroke="#B7B7B7" stroke-width="1.5"/>')
    for g, (px, py) in person_pos.items():
        for rid, (cx, cy) in pos.items():
            if groups[rid] != g:
                continue
            start = border((cx, cy), (px, py))
            end = (px - person_box[0] / 2, py)
            svg.append(f'<line x1="{start[0]:.1f}" y1="{start[1]:.1f}" x2="{end[0]:.1f}"'
                       f' y2="{end[1]:.1f}" stroke="#0091EA" stroke-width="1.2"'
                       f' stroke-dasharray="4 3"/>')
    for rect, (lx, ly), label in placed:
        svg.append(f'<rect x="{rect[0]:.1f}" y="{rect[1]:.1f}" width="{LW}" height="{LH}"'
                   f' rx="4" fill="#FFFFFF" stroke="#B7B7B7"/>')
        svg.append(f'<text x="{lx:.1f}" y="{ly + 4:.1f}" font-size="11" font-weight="bold"'
                   f' fill="#607D8B" text-anchor="middle">{label}</text>')
    for g, (px, py) in person_pos.items():
        pw, ph = person_box
        label, real = persons[g]["label"], persons[g]["real"]
        colour = PERSON_FILLS[sorted({groups[r] for r in pos}).index(g) % len(PERSON_FILLS)]
        # a counterfactual entity is drawn hollow: it is not in the graph
        skin = ('fill="%s" stroke="#FFFFFF"' % colour if real
                else 'fill="#FFFFFF" stroke="%s" stroke-width="1.5" stroke-dasharray="5 3"' % colour)
        svg.append(f'<rect x="{px - pw / 2}" y="{py - ph / 2}" width="{pw}" height="{ph}"'
                   f' rx="6" {skin}/>')
        ink = "#FFFFFF" if real else colour
        svg.append(f'<text x="{px}" y="{py - 3}" font-size="12" font-weight="bold"'
                   f' fill="{ink}" text-anchor="middle">{label}</text>')
        svg.append(f'<text x="{px}" y="{py + 12}" font-size="10" fill="{ink}"'
                   f' text-anchor="middle">{"Person" if real else "Person, not in the graph"}</text>')
    for rid, (cx, cy) in pos.items():
        rect = box(cx, cy)
        svg.append(f'<rect x="{rect[0]}" y="{rect[1]}" width="{BW}" height="{BH}" rx="6"'
                   f' fill="{fill.get(rid, "#CFD8DC")}" stroke="#FFFFFF"/>')
        svg.append(f'<text x="{cx}" y="{cy - 3}" font-size="12" font-weight="bold"'
                   f' fill="#263238" text-anchor="middle">{RECORDS[rid]}</text>')
        svg.append(f'<text x="{cx}" y="{cy + 12}" font-size="10" fill="#607D8B"'
                   f' text-anchor="middle">·{rid}</text>')
    return "\n".join(svg) + "</svg>"

display(SVG(draw()))


Out of 361,694 records, the narrowing step proposed **four**. That is where
the sixty-five billion went.

Two are `Leo Appelbaum` at 1.000 — one person, recorded twice.

Two are `Lee D Applbaum`, at 0.737 and 0.700. Different first name, different
spelling. **The threshold of 0.695 lets both through**; a threshold of 0.75
would not.

Now read the columns the matcher never saw. Leo is president and secretary of
`A LAVA & SONS` and `LAVA WOOD PRODUCTS`, **both at 4800 S Kilbourn** — one
family timber business in one shed, licensed without a break from 2002 to 2021.
Lee is vice-president of two `FOOTACTION` stores, on 87th and on 63rd, two
licence terms each, both lapsed in 2004. Two coherent working lives that touch
nowhere. They are not the same man, and that is how you can tell —
not from the score.

So this is exactly the borderline case the previous section was about: the edges
were created, and nothing here decided what becomes of them. That is the next
section.

## From edges to entities

The threshold produced 714,058 similarity edges. Nobody is a person yet: those
edges are a graph, and something has to decide which groups of records become
one person — one entity.

Leo was the clue. Follow it.

In [110]:
show_json(q('''
MATCH (p:PersonRecord)
WHERE p.fullName IN ['Leo Appelbaum', 'Lee D Applbaum', 'Lee D. Applbaum']
RETURN p.fullName        AS record,
       p.wccCommunity    AS connected_components,
       p.louvainFinest   AS louvain_first_level,
       p.louvainCoarsest AS louvain_last_level,
       p.clusterKey      AS key_actually_used
ORDER BY record
'''))

```json
[
  {
    "record": "Lee D Applbaum",
    "connected_components": 36520,
    "louvain_first_level": 151061,
    "louvain_last_level": 151061,
    "key_actually_used": "L151061"
  },
  {
    "record": "Lee D. Applbaum",
    "connected_components": 36520,
    "louvain_first_level": 151061,
    "louvain_last_level": 151061,
    "key_actually_used": "L151061"
  },
  {
    "record": "Leo Appelbaum",
    "connected_components": 36520,
    "louvain_first_level": 9299,
    "louvain_last_level": 151061,
    "key_actually_used": "L9299"
  },
  {
    "record": "Leo Appelbaum",
    "connected_components": 36520,
    "louvain_first_level": 9299,
    "louvain_last_level": 151061,
    "key_actually_used": "L9299"
  }
]
```

Three algorithms, three different answers about the same four records.

**Connected components** put all four in one component. It has no choice: it
follows edges transitively, so a single borderline edge — the 0.700 between
`Leo` and `Lee D. Applbaum` — is enough to weld two clusters together forever.

**Louvain's last level** also merges them.

**Louvain's first level** separates them, and that is the key the graph uses.

So the question is not only *which algorithm*, but *which level of it*.

The same three records, drawn twice. Nothing moves between the two pictures
except the colour, and the colour is the group.

In [ ]:
LEVELS = """
// The group each algorithm put each record in, and the entity it actually
// resolved to -- which exists for one of those groupings and not the other.
MATCH (p:PersonRecord) WHERE p.fullName IN $names
OPTIONAL MATCH (p)-[:RECORD_RESOLVED_TO]->(e:Person)
RETURN p.id AS record, p.wccCommunity AS connected_components,
       p.louvainCoarsest AS louvain_last, p.louvainFinest AS louvain_first,
       e.name AS person
ORDER BY record
"""

GROUPS = {r["record"]: r for r in q(LEVELS, names=NAMES)}

# The same four records and the same six edges, drawn twice. Nothing moves
# between the two pictures except the colour, and the colour is the group.
# On the right, what each grouping makes of them.
display(SVG(draw("connected components, or louvain's last level: one group, one person",
                 groups={r: g["connected_components"] for r, g in GROUPS.items()},
                 persons={next(iter({g["connected_components"] for g in GROUPS.values()})):
                          {"label": "one entity", "real": False}})))

display(SVG(draw("louvain, first level: two groups, and the two people the graph holds",
                 groups={r: g["louvain_first"] for r, g in GROUPS.items()},
                 persons={g["louvain_first"]: {"label": g["person"], "real": True}
                          for g in GROUPS.values()})))


**Left to connected components, or to Louvain's last level, Leo and Lee are one
person.** Both put all four records in a single group — component 1020 for the
first, Louvain community 161096 for the second — and the entity on the right is drawn hollow
because it is not in the graph: that grouping is not the one the pipeline kept.

Louvain's **first** level separates them, and that is the level it did keep.
Two groups, and on the right the two `Person` nodes the graph really holds,
each in the colour of the records that resolved to it. No edge moved between
the two pictures and no score changed. Only the reading of the neighbourhood
did.

So the clustering can overrule the threshold. Here is what it wrote down:

In [109]:
show_json(q('''
MATCH (p:PersonRecord)
WHERE p.fullName IN ['Leo Appelbaum', 'Lee D Applbaum', 'Lee D. Applbaum']
MATCH (p)-[:RECORD_RESOLVED_TO]->(e:Person)
RETURN e.name AS entity, collect(p.fullName) AS records_merged
'''))

```json
[
  {
    "entity": "Leo Appelbaum",
    "records_merged": [
      "Leo Appelbaum",
      "Leo Appelbaum"
    ]
  },
  {
    "entity": "Lee D Applbaum",
    "records_merged": [
      "Lee D Applbaum",
      "Lee D. Applbaum"
    ]
  }
]
```

**Two entities, not one.** Leo and Lee were kept apart — despite the edges the
threshold let through, and despite connected components being ready to weld
them into one.

That is the real reason the threshold matters less than it feels like it
should: **it is not the last stage.** The similarity edges are not the answer,
they are a graph, and what turns that graph into entities is the clustering,
which looks at a whole neighbourhood instead of one pair at a time.

One name came out right. The question is what that same choice does at the
scale of the whole file.

### Why connected components cannot work here

In [111]:
show_json(q('''
// `wccCommunity` is the property name, but what connected components produces
// is a component: everything reachable from everything else by any chain of
// edges. Louvain's output below is a group, and the two are not comparable.
MATCH (p:PersonRecord) WHERE p.wccCommunity IS NOT NULL
WITH p.wccCommunity AS component, count(*) AS members
RETURN count(*) AS components, max(members) AS largest_component,
       sum(CASE WHEN members > 20 THEN 1 ELSE 0 END) AS bigger_than_20
'''))

```json
[
  {
    "communities": 206310,
    "largest": 37824,
    "bigger_than_20": 242
  }
]
```

**37,824 records in a single component.** Not a company, not a family — one
blob, produced by transitive closure over a similarity graph.

The mechanism is worth stating plainly, because it is the same in every
domain: if A resembles B and B resembles C, connected components declares A and
C the same, *even when they are nothing alike*.

One wrong edge does not cost you one wrong merge. It costs you the union of two
clusters, and the damage compounds.

### What Louvain does with that same blob

In [112]:
show_json(q('''
// Two scopes, because the levels behave very differently in each. `wccCommunity
// 24` is the 37,824-record blob from the query above.
//
// The first scope is not the whole file: these properties exist only on the
// 207,885 records of the 361,694 that have at least one similarity edge. A name
// nobody else shares produces no edge, so the clustering never saw it.
MATCH (p:PersonRecord) WHERE p.louvainLevels IS NOT NULL
RETURN 'the whole similarity graph' AS scope, count(*) AS records,
       count(DISTINCT p.louvainFinest)   AS groups_first_level,
       count(DISTINCT p.louvainCoarsest) AS groups_last_level
UNION
MATCH (p:PersonRecord {wccCommunity: 24})
RETURN 'the largest blob' AS scope, count(*) AS records,
       count(DISTINCT p.louvainFinest)   AS groups_first_level,
       count(DISTINCT p.louvainCoarsest) AS groups_last_level
'''))

```json
[
  {
    "scope": "whole file",
    "records": 207885,
    "groups_first_level": 61525,
    "groups_last_level": 52664
  },
  {
    "scope": "the largest blob",
    "records": 37824,
    "groups_first_level": 5192,
    "groups_last_level": 136
  }
]
```

Inside the blob, the same 37,824 records become **5,242 groups** at the first
level and **129** at the last. Across the whole similarity graph the two
levels are much closer — 61,584 against 52,655 — so coarsening does its damage precisely
where the similarity graph is densest, which is where it is also most wrong.

Louvain does not follow chains — it looks for parts of the graph that are more
densely connected inside than outside, and it does it repeatedly, each pass
merging the groups of the pass before. That is why it has levels at all: the
first is the most granular, and every one after it is coarser by construction.

For people, coarser is the wrong direction. A person is a handful of records
that all resemble each other; it is not a neighbourhood. So the graph keeps the
**first** level:

Keeping the last level would have collapsed the blob's 5,242 groups into 129.
Keeping the first is not guaranteed right either, but it is the granularity a
person lives at.

## Organisations: more signals, and a different way to be wrong

People gave you one field to match on. Organisations give you several — both
sources carry an address — and that changes the design completely.

The name stops being the evidence and becomes the **blocking key**, deliberately
loose:

```cypher
-- names are only used to propose candidates: ~0.3, not ~0.65
apoc.text.join([x IN valid_name_words | trim(x) + "~0.3"], " AND ")
...
WHERE simil > 0.3
```

A name similarity of 0.3 would be absurd as a decision. It is not one. The
decision comes from **four rules, each pairing a second signal with a name
requirement**:

```
SAME_ADDRESS    identical address       name >= 0.3
NORM_ADDRESS    address minus the unit  name >= 0.5
SAME_ZIP        same postal code        name >= 0.8
SHARED_PEOPLE   2+ owners in common     name >= 0.5
```

Read the ladder. **The weaker the second signal, the stronger the name must
be.** A shared postcode says almost nothing, so the name has to carry 0.8. An
identical address says a lot, so 0.3 is allowed.

An edge exists only if at least one rule fires, and it records which.

### What the address rule buys, and what it costs

In [ ]:
show_json(q('''
// The pairs that only SAME_ADDRESS lets through, weakest name first, with the
// address that fired the rule -- an organisation can hold several, so this is
// the one the two have in common.
MATCH (a:Organization)-[r:IS_SIMILAR_TO]->(b:Organization)
WHERE r.methods = ['SAME_ADDRESS'] AND r.score < 0.45
WITH a, b, r, [x IN a.addresses WHERE x IN b.addresses] AS shared
RETURN a.name AS one, b.name AS other, round(r.score * 100) / 100 AS name_similarity,
       shared[0] AS address
ORDER BY name_similarity LIMIT 8
'''))

`BEST QUALITY FURNITURE 2, INC.` and `1604 WEST 63RD INC.` are not the same
business. They were merged because a licence records them at one address — and
the second name *is* that address.

**An address identifies a building, not a business.** Offices are shared,
tenants succeed one another, and accountants register dozens of companies at
their own door. The rule treats a strong-looking signal as if it were an
identifier.

### The case that shows it properly

In [ ]:
GROUP = "L28"

MEMBERS = q("""
// One row per member of the group: how it got in, what the city paid it, and
// one of the contracts that did the paying. `weakest` is the lowest-scoring
// edge holding it here -- where a member has a single edge, that edge is the
// whole reason it is in the group.
MATCH (o:Organization)-[:BELONGS_TO_ORG_GROUP]->(g:OrganizationGroup {clusterId: $group})
OPTIONAL MATCH (o)-[s:IS_SIMILAR_TO]-(:Organization)-[:BELONGS_TO_ORG_GROUP]->(g)
WITH o, count(DISTINCT s) AS edges,
     apoc.coll.toSet(apoc.coll.flatten(collect(s.methods))) AS rules,
     round(min(s.score) * 100) / 100 AS weakest
OPTIONAL MATCH (r:ContractRecord)-[:RECORDED_VENDOR]->(o)
RETURN o.name AS member, o.source AS source, edges, rules, weakest,
       count(DISTINCT r.contractId) AS contracts,
       toInteger(sum(coalesce(r.amount, 0))) AS amount,
       left(coalesce(collect(DISTINCT r.name)[0], ''), 46) AS a_contract
ORDER BY amount DESC
""", group=GROUP)

show_json(MEMBERS)


**Seven organisations in one group, and the money is on four of them.** The
three with nothing come from the licence file, which records a business at an
address and never pays it: those rows bring a name and a street, not money.

Now the question the group exists to answer: **how much has Walsh Construction
been awarded?** Ask the group and you get **$2.26 billion**. Ask the largest
single node and you get $1.12 billion, which misses the other spellings of the
same firm. Neither is the answer, and the gap between them is not rounding.

**And the two joint ventures are not the same case at all.** `WALSH-RITEWAY
CONSTRUCTION JV` is held here by a single edge at 0.67 and brings **$126
million** of its own: leave it out and that money disappears, put it in and the
money is credited to Walsh Construction. `Walsh Vinci Transit Community` is
held by a single edge at **0.38** — the weakest in the table — and carries
nothing at all.

So the edge that looks worst costs nothing, and the edge that moves $126
million is the respectable-looking one at 0.67. **The size of a mistake and the
weakness of the signal behind it are unrelated**, which is why "tighten the
rule" is not a plan.


In [ ]:
GROUP_EDGES = q("""
// Every similarity edge inside the group, each pair once. This is the graph the
// clustering was given -- seven records and the nine edges between them.
MATCH (a:Organization)-[r:IS_SIMILAR_TO]-(b:Organization)
MATCH (a)-[:BELONGS_TO_ORG_GROUP]->(g:OrganizationGroup {clusterId: $group})
MATCH (b)-[:BELONGS_TO_ORG_GROUP]->(g)
WHERE a.name < b.name
RETURN DISTINCT a.name AS one, b.name AS other,
       round(r.score * 100) / 100 AS score, r.methods AS rules
ORDER BY score DESC
""", group=GROUP)

# Laid out by hand, because the shape is the point: the four that hold each
# other in the middle, the three that hang off a single edge pushed outward.
# Money and source come from MEMBERS, so the picture cannot drift from the table.
SHORT = {"WALSH CONSTRUCTION COMPANY": ("WALSH CONSTRUCTION COMPANY", (470, 250)),
         "WALSH CONSTRUCTION COMPANY II, LLC": ("WALSH CONSTRUCTION CO II, LLC", (470, 105)),
         "WALSH CONSTRUCTION COMPANY II, LLC.": ("WALSH CONSTRUCTION CO II, LLC.", (790, 175)),
         "WALSH CONSTRUCTION COMPANY OF ILL": ("WALSH CONSTRUCTION CO OF ILL", (790, 320)),
         "WALSH CONSTRUCTIONQ": ("WALSH CONSTRUCTIONQ", (150, 320)),
         "WALSH-RITEWAY CONSTRUCTION JV": ("WALSH-RITEWAY CONSTRUCTION JV", (150, 175)),
         "Walsh Vinci Transit Community": ("Walsh Vinci Transit Community", (790, 40))}
MONEY = {m["member"]: m["amount"] for m in MEMBERS}
WHERE_FROM = {m["member"]: m["source"] for m in MEMBERS}
BEND = {("WALSH CONSTRUCTION COMPANY II, LLC",
         "WALSH CONSTRUCTION COMPANY OF ILL"): 70}      # the one edge that would cross

W, H, BW, BH = 1000, 404, 184, 54

def cash(n):
    if not n:
        return "no contracts"
    return f"${n / 1e9:.2f}bn" if n >= 1e9 else f"${n / 1e6:.1f}M"

def rim(centre, towards, bend=0):
    """leave a box at its edge, aimed at the other node"""
    cx, cy = centre
    dx, dy = towards[0] - cx, towards[1] - cy + bend
    if not dx and not dy:
        return centre
    step = min((BW / 2) / abs(dx) if dx else 9e9, (BH / 2) / abs(dy) if dy else 9e9)
    return cx + dx * step, cy + dy * step

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       f'<text x="{W / 2}" y="22" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">seven records, nine edges, and three members held'
       f' by one edge each</text>']
for e in GROUP_EDGES:
    a, b = SHORT[e["one"]][1], SHORT[e["other"]][1]
    bend = BEND.get((e["one"], e["other"]), 0)
    p, r = rim(a, b, bend), rim(b, a, bend)
    mx, my = (p[0] + r[0]) / 2, (p[1] + r[1]) / 2 + bend / 2
    # orange where a single weak edge is the whole reason a member is in here
    alone = e["score"] < 0.7
    colour = "#FF9900" if alone else "#B7B7B7"
    svg.append(f'<path d="M {p[0]:.0f} {p[1]:.0f} Q {mx:.0f} {my:.0f},'
               f' {r[0]:.0f} {r[1]:.0f}" fill="none" stroke="{colour}"'
               f' stroke-width="{2.4 if alone else 1.3}"/>')
    label = f'{e["score"]:.2f}  {"+".join(m.split("_")[0].lower() for m in e["rules"])}'
    half = len(label) * 2.5
    svg.append(f'<rect x="{mx - half:.0f}" y="{my - 7:.0f}" width="{half * 2:.0f}"'
               f' height="14" rx="3" fill="#FFFFFF"/>')
    svg.append(f'<text x="{mx:.0f}" y="{my + 3:.0f}" font-size="8.5" font-weight="bold"'
               f' fill="{"#FF9900" if alone else "#607D8B"}" text-anchor="middle">'
               f'{xml(label)}</text>')
for full, (name, (x, y)) in SHORT.items():
    money = MONEY.get(full, 0)
    svg.append(f'<rect x="{x - BW / 2}" y="{y - BH / 2}" width="{BW}" height="{BH}" rx="6"'
               f' fill="{"#CFD8DC" if money else "#ECEFF1"}" stroke="#FFFFFF"/>')
    svg.append(f'<text x="{x}" y="{y - 11}" font-size="9" font-weight="bold" fill="#263238"'
               f' text-anchor="middle">{xml(fit(name, BW, 9))}</text>')
    svg.append(f'<text x="{x}" y="{y + 3}" font-size="8" fill="#607D8B"'
               f' text-anchor="middle">{WHERE_FROM.get(full, "").lower()}</text>')
    svg.append(f'<text x="{x}" y="{y + 18}" font-size="{10 if money else 8.5}"'
               f' font-weight="{"bold" if money else "normal"}"'
               f' fill="{"#0091EA" if money else "#90A4AE"}" text-anchor="middle">'
               f'{cash(money)}</text>')
svg.append(f'<text x="{W / 2}" y="{H - 10}" font-size="10.5" fill="#FF9900"'
           f' text-anchor="middle">orange: the two edges below 0.70. One brings'
           f' {cash(MONEY["WALSH-RITEWAY CONSTRUCTION JV"])} into the group,'
           f' the other brings nothing</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


In [ ]:
# Drawn from MEMBERS, so the picture cannot drift from the table: one band per
# member, its width the square root of the money -- linear widths would make
# the two small flows invisible beside the two large ones.
def money(n):
    return f"${n / 1e9:.2f}bn" if n >= 1e9 else (f"${n / 1e6:.0f}M" if n else "no contracts")

paid = [m for m in MEMBERS if m["amount"]]
total = sum(m["amount"] for m in MEMBERS)
W, H, BW, BH, GAP = 820, 60 + len(MEMBERS) * 44, 290, 34, 44
widest = max(m["amount"] for m in MEMBERS) ** 0.5

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>']
gx, gy, gh = 660, H / 2, 150
for i, m in enumerate(MEMBERS):
    y = 40 + i * GAP
    band = 2 + 24 * (m["amount"] ** 0.5) / widest if m["amount"] else 0
    if band:
        svg.append(f'<path d="M {30 + BW + 96} {y} C {gx - 150} {y}, {gx - 150} {gy}, {gx - 95} {gy}"'
                   f' fill="none" stroke="#ACDBF8" stroke-width="{band:.1f}" stroke-opacity="0.85"/>')
    else:
        svg.append(f'<path d="M {30 + BW + 96} {y} C {gx - 150} {y}, {gx - 150} {gy}, {gx - 95} {gy}"'
                   f' fill="none" stroke="#B7B7B7" stroke-width="1" stroke-dasharray="4 4"/>')
for i, m in enumerate(MEMBERS):
    y = 40 + i * GAP
    svg.append(f'<rect x="30" y="{y - BH / 2}" width="{BW}" height="{BH}" rx="6"'
               f' fill="#CFD8DC" stroke="#FFFFFF"/>')
    svg.append(f'<text x="42" y="{y + 4}" font-size="11.5" font-weight="bold"'
               f' fill="#263238">{xml(m["member"])[:34]}</text>')
    svg.append(f'<text x="{30 + BW + 12}" y="{y + 4}" font-size="11" font-weight="bold"'
               f' fill="{"#0091EA" if m["amount"] else "#9E9E9E"}">{money(m["amount"])}</text>')
svg.append(f'<rect x="{gx - 95}" y="{gy - gh / 2}" width="190" height="{gh}" rx="8"'
           f' fill="#607D8B" stroke="#FFFFFF"/>')
svg.append(f'<text x="{gx}" y="{gy - 16}" font-size="12" font-weight="bold" fill="#FFFFFF"'
           f' text-anchor="middle">OrganizationGroup</text>')
svg.append(f'<text x="{gx}" y="{gy + 2}" font-size="12" fill="#FFFFFF"'
           f' text-anchor="middle">{GROUP}</text>')
svg.append(f'<text x="{gx}" y="{gy + 28}" font-size="16" font-weight="bold" fill="#FFFFFF"'
           f' text-anchor="middle">{money(total)}</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


**Four streams, one box.** In the contracts file those four names are four
recipients, each with contracts of its own; the group makes them one, and its
total is the only number that answers the question asked of it.

The three dashed lines carry no money, and they are not idle: one of them is
the licence spelling whose address is the entire reason the joint venture at
0.38 is in the group at all.


In [ ]:
show_json(q("""
// Where the group's total comes from, contract by contract. `amount` on a
// revision is the change that revision made, not the contract's value at the
// time: the award is the first row by approval date, and everything after it
// is a correction, some of them negative. So the value today is the sum of
// every revision, and summing them is the right arithmetic rather than a
// double count.
MATCH (o:Organization)-[:BELONGS_TO_ORG_GROUP]->(:OrganizationGroup {clusterId: $group})
MATCH (r:ContractRecord)-[:RECORDED_VENDOR]->(o)
WITH r.contractId AS contract, o.name AS member, r
ORDER BY r.approvalDate
WITH contract, member, collect(r.amount) AS amounts, collect(r.approvalDate) AS dates
RETURN contract, member,
       toInteger(amounts[0])                                        AS awarded,
       size(amounts) - 1                                            AS later_revisions,
       toInteger(reduce(a = 0.0, x IN amounts | a + x) - amounts[0]) AS net_change,
       toInteger(reduce(a = 0.0, x IN amounts | a + x))              AS value_now,
       toString(dates[0])                                           AS awarded_on,
       toString(dates[-1])                                          AS last_change
ORDER BY value_now DESC LIMIT 6
""", group=GROUP))


**Read the first row.** The city awarded contract `129033` to `WALSH
CONSTRUCTION COMPANY II, LLC` for **$159.4 million** in May 2020. Since then 177
revisions have carried it to **$213.4 million**, the last one in September
2025. Nobody added $54 million at once: hundreds of adjustments did, some of
them negative, over five years.

`net_change` is therefore a measurement in its own right — **how far a contract
has travelled from the way it was awarded.** Here it is 34%; on `P012016` it is
62%, from $55.2 million to $89.2 million.

Four of these six contracts are recorded against `WALSH CONSTRUCTION COMPANY
OF ILL` and two against `WALSH CONSTRUCTION COMPANY II, LLC`. Every contract
sits on exactly one spelling — checked, none of the 72 is split across two —
so without the group you would read two mid-sized suppliers instead of one
company with a billion dollars of work.

What the table cannot say is **why** any of those revisions exist. The CSV
records that a revision happened and what it changed; the reason is printed on
a Modification Summary Report inside the contract's PDF, which is the other
half of this graph.


## What the entity layer buys

### One owner, four companies, and what it takes to see him

Everything so far has been a decision and its price. This is what the decisions
buy when they both go right.

The question is one a journalist would ask and no single file can answer: **how
much has the city paid to the businesses of one man?** The owners file knows who
owns what and nothing about contracts. The contracts file knows who was paid and
nothing about owners. Only the entity layer joins them — twice over, once for
the person and once for each company.


In [ ]:
PERSON = "L91546"

OWNED = q("""
// One person, and the businesses of his that the city pays. Two resolutions
// are chained here and both have to hold: `RECORD_RESOLVED_TO` says the owner
// records are one human being, and the organisation group says the firm on the
// licence is the firm on the contracts -- `MAT ASHPHALT, LLC` as the licence
// spells it, `MAT Asphalt, LLC.` as the contracts file does.
MATCH (pr:PersonRecord)-[:RECORD_RESOLVED_TO]->(:Person {clusterId: $person})
MATCH (licensed:Organization {id: pr.employerId})-[:BELONGS_TO_ORG_GROUP]->(g:OrganizationGroup)
// the record side is closed here, before the money is touched: he signs for one
// of these firms twice, as secretary and as president, and carrying both rows
// into the sum would pay that company twice.
WITH g, collect(DISTINCT licensed.name)[0] AS on_the_licence,
     collect(DISTINCT pr.title) AS signed_as
MATCH (vendor:Organization {source: 'CONTRACTS'})-[:BELONGS_TO_ORG_GROUP]->(g)
MATCH (r:ContractRecord)-[:RECORDED_VENDOR]->(vendor)
RETURN on_the_licence, collect(DISTINCT vendor.name)[0] AS on_the_contracts,
       g.clusterId AS group, signed_as,
       count(DISTINCT r.contractId) AS contracts,
       toInteger(sum(r.amount)) AS paid_to_the_company
ORDER BY paid_to_the_company DESC
""", person=PERSON)

show_json(OWNED)
print(f"\n{len(OWNED)} companies, "
      f"${sum(r['paid_to_the_company'] for r in OWNED) / 1e6:,.1f} million in all")


In [ ]:
# Drawn from OWNED, in the same idiom as the Walsh flows and read the other way
# round: there four names converged into one group, here one person opens onto
# four. The band is the money, its width the square root of the amount.
person = q("MATCH (p:Person {clusterId: $person}) RETURN p.name AS name",
           person=PERSON)[0]["name"]
W, BH, GAP = 880, 36, 62
H = 80 + len(OWNED) * GAP
widest = max(r["paid_to_the_company"] for r in OWNED) ** 0.5
mid_x, mid_y, mid_h = 400, H / 2, 108

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       f'<text x="{W / 2}" y="26" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">{len(OWNED)} companies, {len(OWNED)} groups, one person —'
       f' and the city pays all four</text>']
for i, r in enumerate(OWNED):
    y = 62 + i * GAP
    svg.append(f'<path d="M 250 {y} C 330 {y}, 330 {mid_y}, {mid_x - 85} {mid_y}"'
               f' fill="none" stroke="#0091EA" stroke-width="1.2" stroke-dasharray="4 3"/>')
    band = 2 + 22 * (r["paid_to_the_company"] ** 0.5) / widest
    svg.append(f'<path d="M {mid_x + 85} {mid_y} C {mid_x + 170} {mid_y}, {mid_x + 170} {y},'
               f' {W - 310} {y}" fill="none" stroke="#ACDBF8" stroke-width="{band:.1f}"'
               f' stroke-opacity="0.85"/>')
for i, r in enumerate(OWNED):
    y = 62 + i * GAP
    svg.append(f'<rect x="30" y="{y - BH / 2}" width="220" height="{BH}" rx="6"'
               f' fill="#CFD8DC" stroke="#FFFFFF"/>')
    svg.append(f'<text x="42" y="{y - 2}" font-size="11" font-weight="bold"'
               f' fill="#263238">{xml(r["on_the_licence"][:28])}</text>')
    svg.append(f'<text x="42" y="{y + 12}" font-size="10" fill="#607D8B">'
               f'{xml(", ".join(s.lower() for s in r["signed_as"]))}</text>')
    svg.append(f'<rect x="{W - 310}" y="{y - BH / 2}" width="230" height="{BH}" rx="6"'
               f' fill="#CFD8DC" stroke="#FFFFFF"/>')
    svg.append(f'<text x="{W - 298}" y="{y - 2}" font-size="11" font-weight="bold"'
               f' fill="#263238">{xml(r["on_the_contracts"][:30])}</text>')
    svg.append(f'<text x="{W - 298}" y="{y + 12}" font-size="10" fill="#607D8B">'
               f'group {r["group"]}</text>')
    svg.append(f'<text x="{W - 68}" y="{y + 3}" font-size="11" font-weight="bold"'
               f' fill="#0091EA">${r["paid_to_the_company"] / 1e6:,.0f}M</text>')
svg.append(f'<rect x="{mid_x - 85}" y="{mid_y - mid_h / 2}" width="170" height="{mid_h}"'
           f' rx="8" fill="#607D8B" stroke="#FFFFFF"/>')
svg.append(f'<text x="{mid_x}" y="{mid_y - 14}" font-size="12" font-weight="bold"'
           f' fill="#FFFFFF" text-anchor="middle">{xml(person)}</text>')
svg.append(f'<text x="{mid_x}" y="{mid_y + 2}" font-size="11" fill="#FFFFFF"'
           f' text-anchor="middle">Person {PERSON}</text>')
svg.append(f'<text x="{mid_x}" y="{mid_y + 26}" font-size="15" font-weight="bold"'
           f' fill="#FFFFFF" text-anchor="middle">'
           f'${sum(r["paid_to_the_company"] for r in OWNED) / 1e6:,.0f}M</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


**$789.6 million, across four companies and four groups.** Read the two name
columns against each other: the licence says `MAT ASHPHALT, LLC`, the contracts
file says `MAT Asphalt, LLC.`; the licence says `M T TRANSIT INC`, the contracts
file says `MOUNT TRANSIT INC`. Neither pair shares an identifier. Each is a
merge the organisation clustering made, and if any of them had gone the other
way the total would be smaller and nobody would notice.

The person side had to hold as well. Nineteen owner records resolve into this
one `Person`, and the four that matter here sign as secretary, president and
managing member of different firms — including one firm he signs for twice,
which is why the query closes the record side before it touches the money.

**Two things this does not say.** It does not say the man received $789.6
million: his companies were paid it, and what reaches him is not in any of
these files. And it does not say he is one man rather than two: every record in
the table is written the same way, and a father and a son of the same name
would be written the same way too. The graph cannot settle that, and neither
can a better threshold.


### A contract still being revised, a licence that ended eleven years ago

The same bridge answers a question about time. A business licence is a term
with a start and an end; a contract revision carries the date it was approved.
They sit in two files that share no identifier, so until the groups existed
there was no way to hold one against the other.


In [ ]:
VENDOR = "ALDRIDGE ELECTRIC INC"

PATH = q("""
// The walk from the latest revision of a contract to the last licence term of
// the same business. Nothing in either file points at the other: the step in
// the middle is the group, which the clustering computed.
MATCH (vendor:Organization {source: 'CONTRACTS', name: $vendor})
      -[:BELONGS_TO_ORG_GROUP]->(group:OrganizationGroup)
      <-[:BELONGS_TO_ORG_GROUP]-(licensed:Organization {source: 'LICENSES'})
MATCH (licensed)-[:ORG_HAS_LICENSE]->(licence:LicenseRecord)
WITH vendor, group, licensed, licence
  ORDER BY right(licence.endDate, 4) + left(licence.endDate, 2) DESC LIMIT 1
MATCH (revision:ContractRecord)-[:RECORDED_VENDOR]->(vendor)
WITH vendor, group, licensed, licence, revision ORDER BY revision.approvalDate DESC LIMIT 1
MATCH (revision)-[:INCLUDED_IN_CONTRACT]->(contract:Contract)
RETURN revision.id AS revision, toString(revision.approvalDate) AS revised_on,
       toInteger(revision.amount) AS revision_amount, contract.id AS purchase_order,
       vendor.id AS vendor_id, group.clusterId AS group, licensed.id AS licensed_id,
       licence.license_id AS licence, licence.startDate AS licence_from,
       licence.endDate AS licence_to
""", vendor=VENDOR)[0]

SCALE = q("""
// And how common the shape is. One row per group that has both sides.
MATCH (lic:Organization {source: 'LICENSES'})-[:BELONGS_TO_ORG_GROUP]->(g:OrganizationGroup)
      <-[:BELONGS_TO_ORG_GROUP]-(ven:Organization {source: 'CONTRACTS'})
MATCH (lic)-[:ORG_HAS_LICENSE]->(l:LicenseRecord)
MATCH (cr:ContractRecord)-[:RECORDED_VENDOR]->(ven)
WITH g, max(date(right(l.endDate, 4) + '-' + left(l.endDate, 2) + '-' +
            substring(l.endDate, 3, 2))) AS licensed_until,
     max(cr.approvalDate) AS revised_until
RETURN count(*) AS groups_with_both_sides,
       sum(CASE WHEN licensed_until < revised_until THEN 1 ELSE 0 END) AS licence_ended_first,
       sum(CASE WHEN licensed_until < revised_until - duration({years: 2})
                THEN 1 ELSE 0 END) AS by_more_than_two_years
""")[0]

show_json([PATH, SCALE])


In [ ]:
# Drawn from PATH: five nodes and the four edges walked between them, in the
# order the query walks them. The two ends carry the dates the case is about.
STEPS = [("ContractRecord", f"·{PATH['revision']}",
          f"{PATH['revised_on']} · ${PATH['revision_amount'] / 1e6:.1f}M"
          f" on PO {PATH['purchase_order']}", True),
         ("Organization", VENDOR[:24], PATH["vendor_id"], False),
         ("OrganizationGroup", PATH["group"], "computed, not loaded", False),
         ("Organization", VENDOR[:24], PATH["licensed_id"], False),
         ("LicenseRecord", f"·{PATH['licence']}",
          f"{PATH['licence_from']} → {PATH['licence_to']}", True)]
EDGES = [("RECORDED_VENDOR", "right"), ("BELONGS_TO_ORG_GROUP", "right"),
         ("BELONGS_TO_ORG_GROUP", "left"), ("ORG_HAS_LICENSE", "right")]

def two_lines(name):
    """the relationship name over two lines, so it fits between two boxes"""
    head, _, tail = name.partition("_")
    return (head, tail) if len(name) > 15 else (name, "")

BW, BH, GAP = 168, 72, 96
W, H = 5 * BW + 4 * GAP + 60, 200
svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       '<defs><marker id="ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6"'
       ' markerHeight="6" orient="auto"><path d="M0 0L10 5L0 10z" fill="#B7B7B7"/></marker>'
       '</defs>']
y = 86
for i, (label, line2, line3, lit) in enumerate(STEPS):
    x = 30 + i * (BW + GAP)
    if i:
        x0, x1 = x - GAP + 4, x - 4
        if EDGES[i - 1][1] == "right":
            svg.append(f'<line x1="{x0}" y1="{y}" x2="{x1}" y2="{y}" stroke="#B7B7B7"'
                       f' stroke-width="1.5" marker-end="url(#ar)"/>')
        else:
            svg.append(f'<line x1="{x1}" y1="{y}" x2="{x0}" y2="{y}" stroke="#B7B7B7"'
                       f' stroke-width="1.5" marker-end="url(#ar)"/>')
        head, tail = two_lines(EDGES[i - 1][0])
        svg.append(f'<text x="{(x0 + x1) / 2:.0f}" y="{y - 22}" font-size="9"'
                   f' font-weight="bold" fill="#607D8B" text-anchor="middle">{head}</text>')
        if tail:
            svg.append(f'<text x="{(x0 + x1) / 2:.0f}" y="{y - 12}" font-size="9"'
                       f' font-weight="bold" fill="#607D8B" text-anchor="middle">{tail}</text>')
    svg.append(f'<rect x="{x}" y="{y - BH / 2}" width="{BW}" height="{BH}" rx="6"'
               f' fill="{"#ACDBF8" if lit else "#CFD8DC"}" stroke="#FFFFFF"/>')
    svg.append(f'<text x="{x + BW / 2}" y="{y - 16}" font-size="10" fill="#607D8B"'
               f' text-anchor="middle">{label}</text>')
    svg.append(f'<text x="{x + BW / 2}" y="{y + 2}" font-size="12" font-weight="bold"'
               f' fill="#263238" text-anchor="middle">{xml(line2)}</text>')
    svg.append(f'<text x="{x + BW / 2}" y="{y + 20}" font-size="9.5" fill="#607D8B"'
               f' text-anchor="middle">{xml(line3)}</text>')
svg.append(f'<text x="{W / 2}" y="30" font-size="13" font-weight="bold" fill="#0091EA"'
           f' text-anchor="middle">a revision approved in {PATH["revised_on"][:4]},'
           f' a licence that ended in {PATH["licence_to"][-4:]}</text>')
svg.append(f'<text x="{W / 2}" y="{H - 26}" font-size="11" fill="#607D8B"'
           f' text-anchor="middle">the two ends come from two files that share no'
           f' identifier; the step in the middle is the one nobody wrote down</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


**Eleven years and nine months.** `ALDRIDGE ELECTRIC INC` holds seven licence
terms in the licence file, the last of them ending in September 2014. In the
contracts file the same name is still being revised in June 2026 — the
revision drawn above is $1.5 million on purchase order 32520.

Read the path itself. The two ends are records, loaded and unmodified; the
step in the middle is `L16309`, a group **nobody wrote down** — the clustering
computed it. Here it is not guessing: the licence and the contracts carry the
same name, letter for letter, and the group only says out loud what the two
files already spell the same way.

**336 of the 1,860 groups that have both sides look like this**, and 233 of
them by more than two years.


### The ring: a contract that comes back to the department that awarded it

The last question is the one the whole entity layer was built for, and the one
no file can be asked: **does any of this money end up with the people who hand
it out?**


In [ ]:
RING = q("""
// A ring: the department awards a contract, the contract names a vendor, the
// vendor shares a group with a licensed business, and the person who holds
// office in that business turns out to be on the department's own payroll.
// Seven hops, and the one that closes it is the group -- which nobody wrote.
MATCH (dept:Department)-[:ASSIGNS_CONTRACT]->(contract:Contract {id: $po})
      -[:HAS_VENDOR]->(vendor:Organization)-[:BELONGS_TO_ORG_GROUP]->(grp:OrganizationGroup)
      <-[:BELONGS_TO_ORG_GROUP]-(licensed:Organization)
MATCH (person:Person)-[:WORKS_FOR_ORG]->(licensed)
MATCH (staff:PersonRecord {source: 'EMPLOYEES'})-[:RECORD_RESOLVED_TO]->(person)
MATCH (staff)-[:WORKS_FOR_DEPARTMENT]->(dept)
MATCH (owner:PersonRecord {source: 'OWNERS'})-[:RECORD_RESOLVED_TO]->(person)
WHERE owner.employerId = licensed.id
MATCH (cr:ContractRecord)-[:RECORDED_VENDOR]->(vendor)
RETURN dept.id AS department, contract.id AS contract, contract.name AS subject,
       toInteger(sum(cr.amount)) AS amount, toString(min(cr.approvalDate)) AS approved,
       vendor.name AS vendor, grp.clusterId AS group, licensed.name AS licensed,
       person.name AS person, person.clusterId AS person_id,
       staff.fullName AS staff, staff.title AS staff_title,
       owner.fullName AS owner, owner.title AS owner_title
""", po="69570")[0]

show_json(RING)


In [ ]:
import math

# Drawn from RING, one box per hop, in the order the query walks them. The ring
# is the point: the money leaves the department and comes back to somebody on
# its payroll. The dashed step is the one that is not an edge at all.
def fit(text, width, size):
    """shorten to what the box can hold, at roughly 0.55 em per character"""
    room = int((width - 14) / (size * 0.55))
    return text if len(text) <= room else text[:room - 1] + "…"

NODES = [("Department", RING["department"], ""),
         ("Contract", RING["contract"], f"${RING['amount']:,} · {RING['approved']}"),
         ("Organization", RING["vendor"], "the contracts file"),
         ("OrganizationGroup", RING["group"], "computed, not loaded"),
         ("Organization", RING["licensed"], "the licence file"),
         ("PersonRecord", RING["owner"], RING["owner_title"].lower()),
         ("Person", RING["person"], RING["person_id"]),
         ("PersonRecord", RING["staff"], RING["staff_title"].lower())]
EDGES = [("ASSIGNS_CONTRACT", 0, 1), ("HAS_VENDOR", 1, 2), ("BELONGS_TO_ORG_GROUP", 2, 3),
         ("BELONGS_TO_ORG_GROUP", 4, 3), ("employerId", 5, 4), ("RECORD_RESOLVED_TO", 5, 6),
         ("RECORD_RESOLVED_TO", 7, 6), ("WORKS_FOR_DEPARTMENT", 7, 0)]

W, H, BW, BH, RX, RY = 1000, 620, 212, 54, 318, 206
cx, cy = W / 2, H / 2 + 10
pos = [(cx + RX * math.sin(2 * math.pi * i / len(NODES)),
        cy - RY * math.cos(2 * math.pi * i / len(NODES))) for i in range(len(NODES))]

def edge_point(a, b):
    """from a box centre out to its own border, aimed at the other box"""
    dx, dy = b[0] - a[0], b[1] - a[1]
    step = min((BW / 2 + 6) / abs(dx) if dx else 9e9, (BH / 2 + 6) / abs(dy) if dy else 9e9)
    return a[0] + dx * step, a[1] + dy * step

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       '<defs><marker id="rr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6"'
       ' markerHeight="6" orient="auto"><path d="M0 0L10 5L0 10z" fill="#B7B7B7"/></marker>'
       '</defs>',
       f'<text x="{cx}" y="{cy - 10}" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">the money leaves the department</text>',
       f'<text x="{cx}" y="{cy + 10}" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">and comes back to its own payroll</text>']
for name, i, j in EDGES:
    p, r = edge_point(pos[i], pos[j]), edge_point(pos[j], pos[i])
    dash = ' stroke-dasharray="5 3"' if name == "employerId" else ''
    svg.append(f'<line x1="{p[0]:.0f}" y1="{p[1]:.0f}" x2="{r[0]:.0f}" y2="{r[1]:.0f}"'
               f' stroke="#B7B7B7" stroke-width="1.5" marker-end="url(#rr)"{dash}/>')
    # the label is pushed outward along the radius until it clears every box
    mx, my = (p[0] + r[0]) / 2, (p[1] + r[1]) / 2
    half = len(name) * 2.9 + 5
    for push in (30, 48, 66, 84, 102):
        out = 1 + push / max(math.hypot(mx - cx, my - cy), 1)
        lx, ly = cx + (mx - cx) * out, cy + (my - cy) * out
        clear = all(abs(lx - bx) > BW / 2 + half or abs(ly - by) > BH / 2 + 10
                    for bx, by in pos)
        if clear:
            break
    svg.append(f'<rect x="{lx - half:.0f}" y="{ly - 8:.0f}"'
               f' width="{len(name) * 5.8 + 10:.0f}" height="15" rx="3" fill="#FFFFFF"/>')
    svg.append(f'<text x="{lx:.0f}" y="{ly + 3:.0f}" font-size="8.5" font-weight="bold"'
               f' fill="#607D8B" text-anchor="middle">{name}</text>')
for (label, line2, line3), (x, y) in zip(NODES, pos):
    lit = label == "PersonRecord"
    svg.append(f'<rect x="{x - BW / 2:.0f}" y="{y - BH / 2:.0f}" width="{BW}" height="{BH}"'
               f' rx="6" fill="{"#ACDBF8" if lit else "#CFD8DC"}" stroke="#FFFFFF"/>')
    svg.append(f'<text x="{x:.0f}" y="{y - 12:.0f}" font-size="9.5" fill="#607D8B"'
               f' text-anchor="middle">{label}</text>')
    svg.append(f'<text x="{x:.0f}" y="{y + 4:.0f}" font-size="11.5" font-weight="bold"'
               f' fill="#263238" text-anchor="middle">{xml(fit(line2, BW, 11.5))}</text>')
    if line3:
        svg.append(f'<text x="{x:.0f}" y="{y + 19:.0f}" font-size="9" fill="#607D8B"'
                   f' text-anchor="middle">{xml(fit(line3, BW, 9))}</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


**The ring closes.** The Department of Planning and Development awarded
$44,153 to `ROMAINE EMPIRE, LLC DBA FARMER'S FRIDGE` in December 2017. The
licence file records a `ROMAINE EMPIRE LLC` whose secretary is a `Justin
Petersen`, and the payroll records a `Justin Petersen` who is a supervising
planner **in that same department**.

Every hop comes from the four CSVs — the title and the department from the
payroll, the office held from the owners file, the award from the contracts
file. The documents contribute nothing here: the corpus begins in 2022 and
this contract is from 2017, so there is not a single page about it.

**This is a lead, not a finding, and the difference matters.** The step that
closes the ring is a person resolution built on names alone — the same
machinery that spent the first half of today being shown wrong. A name is not
an identity, and on this path the graph has nothing stronger to offer.

So what the module produces is not an accusation. It is a question with a path
attached — department, contract, vendor, group, licensed business, office,
person, payroll — short enough to check by hand, and specific enough that
checking it is worth somebody's afternoon. **That is what an entity layer is
for**, and it is also why every decision behind it had to be examined first.


## The other half: how the documents were read

The contracts file publishes two things about the same contract: a row, and the
PDF it was signed on. Everything so far came from rows. This is how the pages
were turned into the 14,494 records that sit beside them.

Four readers were put on the same 300 pages and compared against a hand reading
of those pages. The table is the whole argument:

| reader | what it is | agreement with the hand reading |
|---|---|---|
| regex detectors | a pattern per category of page | **25%** (55 of 220) |
| model, open schema | the model picks the type, the entities and the predicates | **55%** (162 of 297) |
| model, closed schema | the model fills slots we choose, null where the page is silent | used in production |
| parser on fixed forms | values read off the labels printed on the form | used in production |

Those agreements were measured once, on 300 pages read by hand, and they do
not come out of this graph: the comparison is `tools/chicago/corpus/compare_labels.py`, and
the numbers are written up in `docs/ESTRAZIONE-CORPUS.md` §7.1.

The open schema lost on a number that has nothing to do with accuracy: **837
distinct predicates for 1,786 relations** — a new kind of edge every other
fact. Across 124,416 distinct pages that is not a graph, it is well-cited noise.

What decides between the other two is **the layout, not the importance of the
page**. Where the form is fixed, a parser reads the values off their printed
labels: no model, no GPU, 30,927 rows in 9.6 seconds, and re-running it gives
the same answer. Where the layout is free, the model fills a closed set of
slots. The two fail differently — the parser cannot read a value that has
drifted away from its label, the model cannot recognise an absence — and that
is exactly why neither is used for the other's pages.


In [ ]:
# Two pages, two kinds of form. Change DEMO to read the other one: the slots
# travel with the page, because a closed schema is a set of boxes we chose for
# that kind of document and nothing else.
DEMOS = {
    "schedule_d": {
        "page": "contracts:page:166835_r0_p139",
        "what": "a Schedule D compliance plan, in which the prime contractor lists the "
                "certified firms it commits to and the dollar value of each commitment",
        "slots": {
            "prime": "name of the prime contractor making the affidavit",
            "goal_percent": "the contract specific goal stated on the page, or null",
            "participants": "list of {firm, certification, work, amount, percent} for every "
                            "certified firm listed with a commitment; empty list if the form is blank",
            "total_amount": "the total direct participation amount, verbatim, or null",
            "total_percent": "the total direct participation percentage, verbatim, or null",
        },
    },
    "compliance_audit": {
        "page": "contracts:page:187342_r3_p3",
        "what": "a B2Gnow compliance audit page, reporting what a contract committed "
                "to in MBE/WBE participation against what was actually paid",
        "slots": {
            "prime": "name of the prime contractor",
            "goal_percent": "the contract participation goal, verbatim",
            "credit_percent": "the participation actually credited, verbatim",
            "award": "current award amount, verbatim",
            "total_paid": "total paid to date, verbatim",
            "status": "any stated compliance status, e.g. 'Not Meeting Goal'",
            "period": "the contract period shown, verbatim",
        },
    },
}

DEMO = DEMOS["schedule_d"]

PAGE = q("MATCH (p:Page {id: $id}) RETURN p.text AS text, p.nChars AS chars",
         id=DEMO["page"])[0]

print(f"{DEMO['page']} -- {PAGE['chars']} characters of OCR, as pdftotext left them\n")
print(PAGE["text"][:760])


In [ ]:
import os

import requests

# The served model, reached the way the pipeline reaches it: an OpenAI-shaped
# endpoint, temperature 0, the answer forced to be a JSON object. The facts in
# the graph were produced by `corpus/extract_typed.py` with its own versioned
# prompt -- decision 25 -- and this cell asks the same question so you can watch
# it being asked. If the endpoint is down the cell prints the answer of
# 21/10/2026 instead, which is the one the graph was built from.
LLM_BASE = os.environ.get("KGTP_LLM_BASE_URL", "https://kgtp-llm.ngrok-free.app/v1")
LLM_KEY = os.environ.get("KGTP_LLM_API_KEY", "teacher-key")
LLM_MODEL = os.environ.get("KGTP_LLM_MODEL", "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8")

SYSTEM = """You transcribe one page of a City of Chicago contract document into a fixed form.

The page is {what}.

Return ONLY a JSON object with exactly these keys and no others:
{slots}
plus "filled": true if the form on this page has been completed, false if it is a blank template.

Rules, in order of importance:
1. Copy values verbatim from the page. Never normalise, reformat or complete them.
2. Any slot the page does not fill in is null. A blank line, an underscore row, "TBD"
   or "N/A" is null. Returning null is always better than returning a guess.
3. Never carry a value over from the running header at the top of the page unless the
   slot asks for it.
4. Do not add keys. Do not explain."""

# What that page actually put in the graph, for the comparison below.
IN_THE_GRAPH = q("""
MATCH (sub:Organization)-[r:SUBCONTRACTS_FOR]->(prime:Organization)
WHERE r.doc + '_p' + toString(r.page) = split($id, ':')[2]
RETURN sub.name AS subcontractor, prime.name AS prime, r.percent AS percent,
       r.amount AS amount, r.doc AS doc, r.page AS page
""", id=DEMO["page"])

try:
    answer = requests.post(
        f"{LLM_BASE}/chat/completions",
        headers={"Authorization": f"Bearer {LLM_KEY}"},
        json={"model": LLM_MODEL, "temperature": 0, "max_tokens": 1200,
              "response_format": {"type": "json_object"},
              "messages": [{"role": "system",
                            "content": SYSTEM.format(
                                what=DEMO["what"],
                                slots="\n".join(f'  "{k}": {v}' for k, v in DEMO["slots"].items()))},
                           {"role": "user", "content": PAGE["text"][:8000]}]},
        timeout=90).json()["choices"][0]["message"]["content"]
    print("read just now, by the model:\n")
    show_json(answer)
except Exception as error:
    print(f"the endpoint did not answer ({type(error).__name__}), so this is not live.")
    print("what it answered on 21/10/2026 is what the graph holds, below.\n")
    show_json(IN_THE_GRAPH)


In [ ]:
# The same page, as it ended up in the graph: two edges, each carrying the
# document and the page it was read from.
show_json(IN_THE_GRAPH)


**Two edges, and each one carries the page it was read from.** `doc` and
`page` are on the relationship, so every fact the documents contributed can be
opened back to the paper that says it — which is what makes it checkable
rather than merely asserted.

Read the prime's name in the graph: `M.A.T. Leasing,Inc.`, with the space
missing after the comma. The page has it that way and **the model copied it**.
The same page says `MBEMBE Utilization` and `MBE/VVBE goals`, and those were
copied too wherever they were asked for. That is the instruction working as
intended: a model that tidied `MBE/VVBE` into `MBE/WBE` would also, sooner or
later, tidy a number.

One judgement is taken away from it. Whether a form is **filled in or blank**
is recomputed in code from the slots that came back, because that is the thing
the model is worst at: on 29 blank pages in the sample it recognised **none**,
answering each one with a confident form. Its own verdict is kept beside the
computed one, under a name that says whose it is.


### How one contract was divided

The CSV says what a contract is worth. It does not say who the work was
promised to: that is written on the Schedule C and D forms bound into the PDF,
one page per promise. Hold the two against each other and a contract stops
being a single number.

In [ ]:
PO = "357467"

show_json(q("""
// The contract as the CSV holds it, and nothing else: who it was awarded to,
// by whom, for how much. `recipients` is the number of organisations the file
// attaches to this contract -- there is no column for anybody else.
MATCH (c:Contract {id: $po})<-[:INCLUDED_IN_CONTRACT]-(cr:ContractRecord)
OPTIONAL MATCH (c)-[:HAS_VENDOR]->(vendor:Organization)
OPTIONAL MATCH (department:Department)-[:ASSIGNS_CONTRACT]->(c)
RETURN c.id AS contract, left(c.name, 52) AS work,
       collect(DISTINCT vendor.name)[0] AS awarded_to,
       count(DISTINCT vendor) AS recipients,
       collect(DISTINCT department.id)[0] AS department,
       count(DISTINCT cr) AS revisions, toInteger(sum(cr.amount)) AS value,
       toString(min(cr.approvalDate)) AS approved
""", po=PO))


**One line, one recipient.** The city awarded $18.8 million to
`BLINDERMAN CONSTRUCTION COMPANY INC` for work at the airport, and that is the
whole of what the contracts file has to say about who does it. `recipients` is
1 because there is no second column: a subcontractor has nowhere to be written
down.

Now open the PDF of the same contract.

In [ ]:
SPLIT = q("""
// The two halves of one contract. The value comes from the CSV, one row per
// revision; the commitments come from the Schedule C and D forms inside the
// PDF, one edge per promise, each carrying the page it was promised on.
MATCH (c:Contract {id: $po})<-[:INCLUDED_IN_CONTRACT]-(cr:ContractRecord)
WITH c, toInteger(sum(cr.amount)) AS contract_value
MATCH (sub:Organization)-[r:SUBCONTRACTS_FOR]->(prime:Organization)
WHERE r.poNumber = $po AND r.amount IS NOT NULL
RETURN left(c.name, 46) AS work, contract_value, prime.name AS prime,
       sub.name AS subcontractor, toInteger(toFloat(r.amount)) AS committed,
       r.doc AS doc, r.page AS page
ORDER BY committed DESC
""", po=PO)

show_json(SPLIT)

VALUE = SPLIT[0]["contract_value"]
# One commitment is read twice -- the prime lists it on its Schedule D and the
# firm itself files a Schedule C -- so the promises are counted once per page
# and summed once per firm.
PROMISED = {}
for row in SPLIT:
    PROMISED[row["subcontractor"].split(" Contract Staffing")[0]] = row["committed"]
print(f"\ncontract {PO}: ${VALUE:,}")
print(f"promised to {len(PROMISED)} subcontractors: ${sum(PROMISED.values()):,}"
      f"  ({100 * sum(PROMISED.values()) / VALUE:.0f}%)")
print(f"the rest stays with {SPLIT[0]['prime']}: ${VALUE - sum(PROMISED.values()):,}")


In [ ]:
# Drawn from SPLIT: the contract on the left, every promise made out of it on
# the right, the band in proportion to the money. The grey band at the bottom
# is what no form promises away.
def money(n):
    return f"${n / 1e6:.2f}M" if n >= 1e6 else f"${n / 1e3:.0f}k"

KEPT = VALUE - sum(PROMISED.values())
SHARES = sorted(PROMISED.items(), key=lambda kv: -kv[1]) + [(SPLIT[0]["prime"], KEPT)]
W, BH, GAP = 980, 34, 46
H = 70 + len(SHARES) * GAP
widest = max(v for _, v in SHARES) ** 0.5
left_x, right_x, mid = 40, W - 330, 300

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       f'<text x="{W / 2}" y="26" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">one contract from the CSV, its promises from the PDF</text>']
for i, (name, amount) in enumerate(SHARES):
    y = 58 + i * GAP
    keeper = i == len(SHARES) - 1
    band = 2 + 22 * (amount ** 0.5) / widest
    svg.append(f'<path d="M {mid} {H / 2} C {mid + 120} {H / 2}, {mid + 120} {y},'
               f' {right_x - 10} {y}" fill="none" stroke="{"#CFD8DC" if keeper else "#ACDBF8"}"'
               f' stroke-width="{band:.1f}" stroke-opacity="0.9"/>')
    svg.append(f'<rect x="{right_x}" y="{y - BH / 2}" width="250" height="{BH}" rx="6"'
               f' fill="{"#ECEFF1" if keeper else "#CFD8DC"}" stroke="#FFFFFF"/>')
    svg.append(f'<text x="{right_x + 10}" y="{y + 4}" font-size="10.5"'
               f' font-weight="{"normal" if keeper else "bold"}" fill="#263238">'
               f'{xml((name[:30] + "…") if len(name) > 31 else name)}</text>')
    svg.append(f'<text x="{W - 12}" y="{y + 4}" font-size="10.5" font-weight="bold"'
               f' fill="{"#607D8B" if keeper else "#0091EA"}" text-anchor="end">'
               f'{money(amount)}</text>')
svg.append(f'<rect x="{left_x}" y="{H / 2 - 44}" width="260" height="88" rx="8"'
           f' fill="#607D8B" stroke="#FFFFFF"/>')
svg.append(f'<text x="{left_x + 130}" y="{H / 2 - 18}" font-size="11" fill="#FFFFFF"'
           f' text-anchor="middle">contract {PO}</text>')
svg.append(f'<text x="{left_x + 130}" y="{H / 2 + 6}" font-size="17" font-weight="bold"'
           f' fill="#FFFFFF" text-anchor="middle">{money(VALUE)}</text>')
svg.append(f'<text x="{left_x + 130}" y="{H / 2 + 28}" font-size="10" fill="#FFFFFF"'
           f' text-anchor="middle">{100 * sum(PROMISED.values()) / VALUE:.0f}% promised out</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


In [ ]:
show_json(q("""
// Does the graph already know these firms? Matched on the name with
// punctuation and case stripped -- which is itself the resolution step the
// loader was not allowed to take, so a stricter or looser rule would move
// rows between these answers. That is the honest state of the join, not a
// shortcut around it.
MATCH (sub:Organization)-[r:SUBCONTRACTS_FOR]->(:Organization)
WHERE r.poNumber = $po AND r.amount IS NOT NULL
WITH DISTINCT sub, toInteger(toFloat(r.amount)) AS committed,
     toUpper(apoc.text.replace(split(sub.name, ' Contract Staffing')[0],
                               '[^A-Za-z0-9]', '')) AS key
OPTIONAL MATCH (known:Organization)
  WHERE known.source <> 'DOC_ORGANIZATION'
    AND toUpper(apoc.text.replace(known.name, '[^A-Za-z0-9]', '')) = key
WITH sub, committed, collect(DISTINCT known.source) AS sources
RETURN sub.name AS subcontractor, committed,
       CASE WHEN 'CONTRACTS' IN sources THEN 'a vendor the city pays'
            WHEN 'LICENSES'  IN sources THEN 'licensed here, never paid by the city'
            ELSE 'the graph has never seen it' END AS in_the_structured_graph
ORDER BY committed DESC
""", po=PO))


**Forty per cent of contract 357467 was promised away before the work began.**
$18.8 million to Blinderman Construction in July 2026, of which $7.58 million
is committed on paper to seven firms — electrical, mechanical, demolition,
staffing — and $11.21 million stays with the prime.

Look at the two `HorizonWorks` rows in the table. The same $200,000 appears
twice, once on page 243 where the prime lists its commitments and once on page
340 where the firm files its own letter of intent. Both edges are true, each
cites a different page, and counting the promises means deciding that they are
one. **The document layer arrives with the same problem the CSVs had**, one
level down.

**Now ask who those seven firms are, and the second table answers: not one of
them is a vendor.** In the structured graph the whole $18.8 million belongs to
`BLINDERMAN CONSTRUCTION COMPANY INC`, because the contracts file records who
won the bid and has nowhere to put anyone else. Four of the seven exist only
because they hold a business licence, and `Staylitt Electric` and `PCS Power &
Communication Services` — two million dollars each — the graph had never seen
at all.

That is not a gap in the extraction, it is the reason for it: **a subcontractor
reaches the contracts file only if somewhere else it won as a prime.** The
document layer brings a population the structured data cannot contain by
construction.

And the shape is rarer than it looks: of the 62 contracts with subcontract
forms, only eight carry both a value in the CSV and amounts legible enough to
add up. The rest are blank templates or OCR too broken to read — `$ DUR`,
`7o1`, `hgggeç !nc-`. This is a mechanism and an order of magnitude, not a
statistic.

### A contract, and the person who handled it

Until now a contract pointed at a department — `DEPARTMENT OF WATER
MANAGEMENT`, a box on an organisation chart with two thousand people inside it.
The documents name somebody on the page.


In [ ]:
CONFIRMED = q("""
// People the documents name on the city's side of a contract, who the payroll
// then confirms in the department that awarded it. The match is the full name,
// stripped of courtesy titles and punctuation -- which is safe on a rare name
// and is exactly the trap on a common one.
MATCH (named:PersonRecord)-[:ACTED_ON]->(c:Contract)<-[:ASSIGNS_CONTRACT]-(d:Department)
WHERE named.source STARTS WITH 'DOC_' AND named.side = 'city' AND named.fullName IS NOT NULL
WITH named, c, d,
     toUpper(apoc.text.replace(named.fullName, '(?i)^(mr|mrs|ms|dr)[. ]+|[^A-Za-z ]', '')) AS key
MATCH (staff:PersonRecord {source: 'EMPLOYEES'})-[:WORKS_FOR_DEPARTMENT]->(d)
WHERE toUpper(apoc.text.replace(staff.fullName, '[^A-Za-z ]', '')) = key
RETURN count(DISTINCT key) AS people, count(DISTINCT c) AS contracts,
       count(DISTINCT d) AS departments
""")[0]

OFFICER = "Sayeh Amirshaghaghi"

PAYROLL = q("""
MATCH (staff:PersonRecord {source: 'EMPLOYEES', fullName: $name})
      -[w:WORKS_FOR_DEPARTMENT]->(d:Department)
RETURN staff.id AS record, staff.title AS title, d.id AS department,
       toInteger(toFloat(w.annualSalary)) AS salary
""", name=OFFICER)[0]

HANDLED = q("""
// The contracts the documents put her on, with the page and the date of each
// appearance, and what the CSV says the contract is worth.
MATCH (named:PersonRecord {fullName: $name})-[a:ACTED_ON]->(c:Contract)
OPTIONAL MATCH (cr:ContractRecord)-[:INCLUDED_IN_CONTRACT]->(c)
OPTIONAL MATCH (c)-[:HAS_VENDOR]->(v:Organization)
RETURN c.id AS contract, left(c.name, 34) AS work,
       collect(DISTINCT v.name)[0] AS vendor, toInteger(sum(cr.amount)) AS value,
       collect(DISTINCT a.doc)[0] AS document, collect(DISTINCT a.page)[0] AS page,
       collect(DISTINCT a.date)[0] AS named_on
ORDER BY value DESC
""", name=OFFICER)

show_json(CONFIRMED)
show_json(PAYROLL)
show_json(HANDLED)
print(f"\n{OFFICER}: {len(HANDLED)} contracts, "
      f"${sum(h['value'] for h in HANDLED):,}, "
      f"{len({h['vendor'] for h in HANDLED})} vendors")


In [ ]:
# Drawn from HANDLED and PAYROLL: the contracts on the left, the person the
# payroll knows on the right, the band in proportion to what the contract is
# worth. Before the documents the left column pointed at a department.
def money(n):
    return f"${n / 1e6:.1f}M"

W, BH, GAP = 1020, 32, 40
H = 90 + len(HANDLED) * GAP
right_x, mid_y = W - 330, (90 + len(HANDLED) * GAP) / 2
widest = max(h["value"] for h in HANDLED) ** 0.5

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       f'<text x="{W / 2}" y="28" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">{len(HANDLED)} contracts named her on the page;'
       f' the payroll says who she is</text>']
for i, h in enumerate(HANDLED):
    y = 62 + i * GAP
    band = 2 + 18 * (h["value"] ** 0.5) / widest
    svg.append(f'<path d="M 470 {y} C 560 {y}, 560 {mid_y}, {right_x - 8} {mid_y}"'
               f' fill="none" stroke="#ACDBF8" stroke-width="{band:.1f}"'
               f' stroke-opacity="0.85"/>')
    svg.append(f'<rect x="30" y="{y - BH / 2}" width="440" height="{BH}" rx="5"'
               f' fill="#CFD8DC" stroke="#FFFFFF"/>')
    svg.append(f'<text x="42" y="{y + 4}" font-size="10" font-weight="bold" fill="#263238">'
               f'{h["contract"]}</text>')
    svg.append(f'<text x="86" y="{y + 4}" font-size="10" fill="#263238">'
               f'{xml(h["work"])}</text>')
    svg.append(f'<text x="352" y="{y + 4}" font-size="9" fill="#607D8B">'
               f'p.{h["page"]} · {h["named_on"]}</text>')
    svg.append(f'<text x="460" y="{y + 4}" font-size="10" font-weight="bold" fill="#0091EA"'
               f' text-anchor="end">{money(h["value"])}</text>')
svg.append(f'<rect x="{right_x}" y="{mid_y - 56}" width="300" height="112" rx="8"'
           f' fill="#607D8B" stroke="#FFFFFF"/>')
svg.append(f'<text x="{right_x + 150}" y="{mid_y - 30}" font-size="13" font-weight="bold"'
           f' fill="#FFFFFF" text-anchor="middle">{xml(OFFICER)}</text>')
svg.append(f'<text x="{right_x + 150}" y="{mid_y - 10}" font-size="10.5" fill="#FFFFFF"'
           f' text-anchor="middle">{xml(PAYROLL["title"].title())}</text>')
svg.append(f'<text x="{right_x + 150}" y="{mid_y + 6}" font-size="9.5" fill="#FFFFFF"'
           f' text-anchor="middle">{xml(PAYROLL["department"].title())}</text>')
svg.append(f'<text x="{right_x + 150}" y="{mid_y + 28}" font-size="10" fill="#FFFFFF"'
           f' text-anchor="middle">payroll record ·{PAYROLL["record"]}'
           f' · ${PAYROLL["salary"]:,}</text>')
svg.append(f'<text x="{right_x + 150}" y="{mid_y + 46}" font-size="11" font-weight="bold"'
           f' fill="#ACDBF8" text-anchor="middle">'
           f'{money(sum(h["value"] for h in HANDLED))} across {len(HANDLED)} contracts</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


**Nine sewer contracts, $58.7 million, five different vendors, and one name on
page 2 of every one of them.** `Sayeh Amirshaghaghi` appears on the
modification report of each, dated from February 2024 to April 2025 — and the
payroll, which is a CSV and knows nothing about any document, says she is the
`CHIEF WATER ENGINEER` of the Department of Water Management on $178,356 a
year. The same department that awarded all nine.

That is the circle closing. The contracts file gave a contract a department;
the payroll gave a department its people; neither connected the two ends. The
page does: it writes a name next to a purchase order, and once that name is
matched the contract is attached to a person with a job title, a salary and a
department — and every step of it can be opened back to the page and the date.

**The match is a name, and that is the whole of its weakness.** It holds here
because `Amirshaghaghi` is rare and because the payroll must put her in the
very department that signed the contract — two conditions, not one. Run the
same query on a common name and the two conditions stop protecting you: you
are back in the ring of the previous section, asserting a person who does not
exist. Across the whole corpus the
two conditions together confirm **19 people on 38 contracts in 10
departments**: a small, checkable set, which is what this kind of evidence
should produce.


In [ ]:
CHAIN = q("""
// The whole path behind one line of the table above, on one document: the page
// the name was read from, the contract that page belongs to, the department
// that awarded it, and the payroll record that puts her in that department.
// `dice` and the two `exists` are the hop that is not there.
MATCH (:ContractRecord)-[:HAS_DOCUMENT]->(d:Document {id: $document})
      -[:HAS_PAGE]->(p:Page)<-[:EXTRACTED_FROM]-(named:PersonRecord {fullName: $name})
MATCH (named)-[:ACTED_ON]->(c:Contract)<-[:ASSIGNS_CONTRACT]-(dept:Department)
MATCH (staff:PersonRecord {source: 'EMPLOYEES'})-[w:WORKS_FOR_DEPARTMENT]->(dept)
WHERE toUpper(staff.fullName) = toUpper(named.fullName)
MATCH (staff)-[:RECORD_RESOLVED_TO]->(person:Person)
RETURN d.poNumber AS po, d.revision AS revision, p.page AS page,
       [line IN split(p.text, '\\n')
        WHERE line CONTAINS 'To:' OR line CONTAINS 'Department of'
           OR line CONTAINS 'Attn:'][0..3] AS quote,
       named.role AS read_as, c.id AS contract, left(c.name, 24) AS work,
       dept.id AS department, staff.id AS payroll, staff.title AS payroll_title,
       toInteger(toFloat(w.annualSalary)) AS salary, person.clusterId AS entity,
       exists { (named)-[:RECORD_RESOLVED_TO]->() } AS doc_record_resolved,
       exists { (named)-[:IS_SIMILAR_TO]-() } AS doc_record_compared,
       round(apoc.text.sorensenDiceSimilarity(named.fullName, staff.fullName) * 1000) / 1000
         AS dice
ORDER BY p.page LIMIT 1
""", document="contracts:doc:149208_r4", name=OFFICER)[0]

# One loop, drawn clockwise: along the top the paper and what was read off it,
# down the right into the graph that was already here, back along the bottom to
# the person the payroll knows. The orange hop is the one that would close it.
# The contract total comes from HANDLED, which already summed its revisions --
# re-summing here would only count the revision this one document carries.
W, H, BW, BH = 1040, 400, 196, 54
TOP, BOT = 118, 300
VALUE = {h["contract"]: h["value"] for h in HANDLED}[CHAIN["contract"]]
BOX = {
    "doc":   (140, TOP, "#ECEFF1", ["Document", f'{CHAIN["po"]}_r{CHAIN["revision"]}',
                                    "the PDF on the city's portal"]),
    "page":  (390, TOP, "#ECEFF1", [f'Page {CHAIN["page"]}', "inter-office communication",
                                    "with a text layer"]),
    "read":  (650, TOP, "#ACDBF8", ["PersonRecord", OFFICER,
                                    f'read as {CHAIN["read_as"]}']),
    "con":   (900, TOP, "#CFD8DC", [f'Contract {CHAIN["contract"]}', CHAIN["work"].strip(),
                                    f'${VALUE:,}']),
    "dept":  (900, BOT, "#CFD8DC", ["Department", "Water Management",
                                    "a box on a chart"]),
    "staff": (650, BOT, "#CFD8DC", ["PersonRecord", OFFICER,
                                    f'{CHAIN["payroll_title"]} · ${CHAIN["salary"]:,}']),
    "ent":   (390, BOT, "#607D8B", ["Person", CHAIN["entity"], "one record, never merged"]),
}
# (from, to, label, exists) -- the arrow runs the way the edge runs in the graph
LINKS = [("doc", "page", "HAS_PAGE", True),
         ("read", "page", "EXTRACTED_FROM", True),
         ("read", "con", "ACTED_ON", True),
         ("dept", "con", "ASSIGNS_CONTRACT", True),
         ("staff", "dept", "WORKS_FOR_DEPARTMENT", True),
         ("staff", "ent", "RECORD_RESOLVED_TO", True),
         ("staff", "read", f'RECORD_RESOLVED_TO · dice {CHAIN["dice"]:.3f}', False)]

def side(a, b):
    """leave a box from the face that looks at the other one"""
    (ax, ay), (bx, by) = BOX[a][:2], BOX[b][:2]
    if ay == by:
        return (ax + (BW / 2 if bx > ax else -BW / 2), ay)
    return (ax, ay + (BH / 2 if by > ay else -BH / 2))

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       '<defs><marker id="cg" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6"'
       ' markerHeight="6" orient="auto"><path d="M0 0L10 5L0 10z" fill="#90A4AE"/></marker>'
       '<marker id="co" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6"'
       ' markerHeight="6" orient="auto"><path d="M0 0L10 5L0 10z" fill="#FF9900"/></marker>'
       '</defs>',
       f'<text x="{W / 2}" y="26" font-size="13" font-weight="bold" fill="#0091EA"'
       f' text-anchor="middle">one page, and the six edges between it and'
       f' {xml(OFFICER)} on the city payroll</text>']
for a, b, label, real in LINKS:
    (x1, y1), (x2, y2) = side(a, b), side(b, a)
    colour = "#90A4AE" if real else "#FF9900"
    dash = "" if real else ' stroke-dasharray="7 5"'
    svg.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{colour}"'
               f' stroke-width="1.5"{dash} marker-end="url(#{"cg" if real else "co"})"/>')
    # A label on a horizontal edge has 54px of gap and needs more, so it goes
    # outside its own row instead: above the top row, below the bottom one.
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    if y1 == y2:
        my += -40 if y1 == TOP else 40
    half = len(label) * 2.55
    svg.append(f'<rect x="{mx - half:.0f}" y="{my - 7:.0f}" width="{half * 2:.0f}"'
               f' height="14" rx="3" fill="#FFFFFF"/>')
    svg.append(f'<text x="{mx:.0f}" y="{my + 3:.0f}" font-size="8.5" font-weight="bold"'
               f' fill="{"#607D8B" if real else "#FF9900"}" text-anchor="middle">'
               f'{xml(label)}</text>')
for key, (x, y, fill, lines) in BOX.items():
    dark = fill == "#607D8B"
    svg.append(f'<rect x="{x - BW / 2}" y="{y - BH / 2}" width="{BW}" height="{BH}" rx="6"'
               f' fill="{fill}" stroke="#FFFFFF"/>')
    for n, (text, size, colour) in enumerate(zip(lines, (11, 10, 8.5),
                                                 ("#263238", "#263238", "#607D8B"))):
        svg.append(f'<text x="{x}" y="{y - 14 + n * 15:.0f}" font-size="{size}"'
                   f' font-weight="{"bold" if n == 0 else "normal"}"'
                   f' fill="{"#FFFFFF" if dark else colour}" text-anchor="middle">'
                   f'{xml(fit(text, BW, size))}</text>')
svg.append(f'<text x="80" y="{(TOP + BOT) / 2 - 26}" font-size="9" font-weight="bold"'
           f' fill="#0091EA">what page {CHAIN["page"]} actually says</text>')
# The page names two people. The line in blue is the one this drawing follows;
# the other, `Alfonzo Conner Jr`, closes the same way on his own payroll record.
for n, line in enumerate(CHAIN["quote"]):
    ours = OFFICER.split()[-1] in line
    svg.append(f'<text x="80" y="{(TOP + BOT) / 2 - 8 + n * 14}" font-size="9.5"'
               f' fill="{"#0091EA" if ours else "#607D8B"}"'
               f' font-weight="{"bold" if ours else "normal"}"'
               f' font-family="monospace">{xml(line.strip())}</text>')
svg.append(f'<text x="{W / 2}" y="{H - 12}" font-size="10.5" fill="#FF9900"'
           f' text-anchor="middle">orange: the hop nobody wrote. The two names score'
           f' {CHAIN["dice"]:.3f} — and the records were never compared'
           f' (compared: {str(CHAIN["doc_record_compared"]).lower()},'
           f' resolved: {str(CHAIN["doc_record_resolved"]).lower()})</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


In [ ]:
SHAPE = q("""
// Every edge that touches the document side, with the labels at both ends.
// `(doc)` marks a node the reading produced; everything else was already here.
MATCH (a)-[r]->(b)
WHERE a.source STARTS WITH 'DOC_' OR b.source STARTS WITH 'DOC_'
   OR a:Document OR a:Page OR b:Document OR b:Page OR a:Claim OR b:Claim
   OR a:Evidence OR b:Evidence
WITH labels(a)[0] + CASE WHEN a.source STARTS WITH 'DOC_' THEN ' (doc)' ELSE '' END AS source,
     type(r) AS edge,
     labels(b)[0] + CASE WHEN b.source STARTS WITH 'DOC_' THEN ' (doc)' ELSE '' END AS target,
     count(*) AS edges
RETURN source, edge, target, edges ORDER BY edges DESC
""")

MISSING = q("""
// And the two that were deliberately not written -- decision 65. Resolving
// them is the exercise, not a step the loader is allowed to take quietly.
MATCH (p:PersonRecord) WHERE p.source STARTS WITH 'DOC_'
WITH count(p) AS people,
     sum(CASE WHEN exists { (p)-[:RECORD_RESOLVED_TO]->() } THEN 1 ELSE 0 END) AS resolved
MATCH (o:Organization {source: 'DOC_ORGANIZATION'})
RETURN people, resolved AS people_resolved, count(o) AS organisations,
       sum(CASE WHEN exists { (o)-[:BELONGS_TO_ORG_GROUP]->() } THEN 1 ELSE 0 END) AS organisations_grouped
""")[0]

show_json(SHAPE)
show_json(MISSING)


In [ ]:
# Drawn from SHAPE and MISSING. Three lanes, read top to bottom: the paper, what
# reading made of it, and the graph that was already here. Edges curve instead
# of cutting across -- provenance rises to the page, joins fall to the graph --
# and dashed orange is an edge that does not exist.
EDGE = {(e["source"], e["edge"], e["target"]): e["edges"] for e in SHAPE}
LANE = {"paper": 110, "read": 300, "graph": 500}
BOX = {"Page": (700, LANE["paper"]), "Document": (1000, LANE["paper"]),
       "Organization (doc)": (180, LANE["read"]), "PersonRecord (doc)": (520, LANE["read"]),
       "Modification (doc)": (860, LANE["read"]),
       "OrganizationGroup": (150, LANE["graph"]), "Person": (430, LANE["graph"]),
       "Contract": (710, LANE["graph"]), "ContractRecord": (1000, LANE["graph"])}
DRAW = [("Document", "HAS_PAGE", "Page", True),
        ("ContractRecord", "HAS_DOCUMENT", "Document", True),
        ("PersonRecord (doc)", "EXTRACTED_FROM", "Page", True),
        ("Modification (doc)", "EXTRACTED_FROM", "Page", True),
        ("PersonRecord (doc)", "FILED_EDS_FOR", "Organization (doc)", True),
        ("Organization (doc)", "DISCLOSED_FOR", "Contract", True),
        ("PersonRecord (doc)", "ACTED_ON", "Contract", True),
        ("Modification (doc)", "MODIFIES_CONTRACT", "Contract", True),
        ("PersonRecord (doc)", "RECORD_RESOLVED_TO", "Person", False),
        ("Organization (doc)", "BELONGS_TO_ORG_GROUP", "OrganizationGroup", False)]

W, H, BW, BH = 1160, 600, 196, 46

def anchor(a, b):
    """leave a box from the side that faces the other one"""
    (ax, ay), (bx, by) = BOX[a], BOX[b]
    if abs(by - ay) > BH:                      # another lane: leave top or bottom
        return (ax, ay + (BH / 2 if by > ay else -BH / 2)), "v"
    return (ax + (BW / 2 if bx > ax else -BW / 2), ay), "h"

svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="100%"'
       f' font-family="Arial, Helvetica, sans-serif">',
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>',
       '<defs><marker id="sg" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6"'
       ' markerHeight="6" orient="auto"><path d="M0 0L10 5L0 10z" fill="#B7B7B7"/></marker>'
       '<marker id="sm" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6"'
       ' markerHeight="6" orient="auto"><path d="M0 0L10 5L0 10z" fill="#FF9900"/></marker>'
       '</defs>']
for lane, title in (("paper", "the paper"), ("read", "what reading made of it"),
                    ("graph", "the graph that was already here")):
    svg.append(f'<text x="28" y="{LANE[lane] - BH / 2 - 14}" font-size="11.5"'
               f' font-weight="bold" fill="#0091EA">{title}</text>')

taken = []
for a, name, b, exists in DRAW:
    (p, kind), (r, _) = anchor(a, b), anchor(b, a)
    if kind == "v":
        c1, c2 = (p[0], (p[1] + r[1]) / 2), (r[0], (p[1] + r[1]) / 2)
    else:
        c1, c2 = ((p[0] + r[0]) / 2, p[1]), ((p[0] + r[0]) / 2, r[1])
    colour, dash = ("#B7B7B7", "") if exists else ("#FF9900", ' stroke-dasharray="6 4"')
    svg.append(f'<path d="M {p[0]:.0f} {p[1]:.0f} C {c1[0]:.0f} {c1[1]:.0f},'
               f' {c2[0]:.0f} {c2[1]:.0f}, {r[0]:.0f} {r[1]:.0f}" fill="none"'
               f' stroke="{colour}" stroke-width="1.4"{dash}'
               f' marker-end="url(#{"sg" if exists else "sm"})"/>')
    text = f'{name}  {EDGE.get((a, name, b), 0):,}'
    half = len(text) * 2.6
    for t in (0.5, 0.36, 0.64, 0.26, 0.74):          # along the curve, until clear
        u = 1 - t
        mx = u**3 * p[0] + 3*u*u*t * c1[0] + 3*u*t*t * c2[0] + t**3 * r[0]
        my = u**3 * p[1] + 3*u*u*t * c1[1] + 3*u*t*t * c2[1] + t**3 * r[1]
        here = (mx - half, my - 7, mx + half, my + 7)
        if all(abs(mx - bx) > BW / 2 + half or abs(my - by) > BH / 2 + 8
               for bx, by in BOX.values()) and \
           all(here[2] < o[0] or o[2] < here[0] or here[3] < o[1] or o[3] < here[1]
               for o in taken):
            break
    else:
        # nowhere along a short edge is clear: lift the label above it instead
        my -= BH / 2 + 10
        here = (mx - half, my - 7, mx + half, my + 7)
    taken.append(here)
    svg.append(f'<rect x="{mx - half:.0f}" y="{my - 7:.0f}" width="{half * 2:.0f}"'
               f' height="14" rx="3" fill="#FFFFFF"/>')
    svg.append(f'<text x="{mx:.0f}" y="{my + 3:.0f}" font-size="8" font-weight="bold"'
               f' fill="{"#607D8B" if exists else "#FF9900"}" text-anchor="middle">{text}</text>')
for label, (x, y) in BOX.items():
    doc = "(doc)" in label
    svg.append(f'<rect x="{x - BW / 2}" y="{y - BH / 2}" width="{BW}" height="{BH}" rx="6"'
               f' fill="{"#ACDBF8" if doc else "#CFD8DC"}" stroke="#FFFFFF"/>')
    svg.append(f'<text x="{x}" y="{y + 5}" font-size="12" font-weight="bold" fill="#263238"'
               f' text-anchor="middle">{xml(fit(label, BW, 12))}</text>')
svg.append(f'<text x="{W / 2}" y="{H - 16}" font-size="11" fill="#FF9900"'
           f' text-anchor="middle">orange: the two edges the loader was not allowed to'
           f' write — {MISSING["people"]:,} people and {MISSING["organisations"]:,}'
           f' organisations still unresolved</text>')
svg.append('</svg>')
display(SVG("\n".join(svg)))


## Close

Everything today was one graph, and almost all of it was about the part nobody
wrote down. Four public files went in; what came out has an entity layer on
top, and that layer is somebody's decision, three times over.

| | the decision | what it costs |
|---|---|---|
| 1 | a similarity threshold of 0.695 | two different men proposed as one, on a score that cannot be shown right |
| 2 | the finest level of Louvain | the level decides who is one person; connected components would have welded 37,824 records into a single blob |
| 3 | four rules, each pairing a second signal with a weaker name | a joint venture worth $126 million credited to its parent company |

The fourth is the same rules refusing on a weak name — a real relation that no
rule can reach. It is the one this session does not show you, and the one to go
looking for.

Then what that layer buys when it holds: one owner reached through four
companies and $789.6 million; a contract still being revised eleven years after
the firm's last licence; a planner on a department's payroll who is secretary
of a company that department pays.

And the other half, which arrived from the documents: a contract that stops
being one number and becomes 40% promised to seven firms the contracts file
never names, and a department that stops being a box on a chart and becomes a
named engineer on nine purchase orders.

**Four things worth taking to a graph that is not this one.**

1. A threshold can be shown wrong, never shown right. Ask which half of the
   question your data can answer before you tune anything.
2. The granularity of a clustering is a decision with a price on both sides,
   and the price is paid in entities nobody inspects.
3. A query that crosses the entity layer inherits every mistake in it. The
   longer the path, the more confident and the more wrong the answer can be —
   `Michael Tadin` and `Justin Petersen` are the same query, one holding and
   one only a lead.
4. A fact you can open is worth more than a fact you can only assert. Every
   edge the documents contributed carries the document and the page, which is
   why a reader can disagree with it.


In [129]:
driver.close()